# Huấn luyện AI kiểm tra bao bì kiện hàng — OK / Lỗi (rách · móp · bẹp-vỡ · ướt)

**Đồ án Smart Logistics Sorter — module AI-01 → AI-07.** Notebook làm trọn quy trình từ thu thập dữ liệu đến file ONNX chạy trên Raspberry Pi 4:

1. **Thu thập dữ liệu từ 5 nguồn:** ảnh nhóm tự chụp tại trạm (quan trọng nhất), Roboflow Universe, Kaggle, Hugging Face Hub, cào ảnh web (Bing, DuckDuckGo).
2. **Chuẩn hóa và làm sạch:** gộp nhãn của mọi nguồn về một hệ chung, bỏ ảnh hỏng/quá nhỏ/một màu, khử trùng lặp gần đúng bằng perceptual hash, lọc ảnh web sai chủ đề bằng CLIP.
3. **Chia tập theo nhóm:** ảnh của cùng một hộp (hoặc các bản augment của cùng một ảnh gốc) không bao giờ nằm ở cả train và test.
4. **Tăng cường dữ liệu:** biến đổi hình học, ánh sáng, nhiễu, nén JPEG + **sinh lỗi tổng hợp** (vẽ vết rách, vết móp, góc bẹp, vệt ướt lên hộp lành).
5. **Mô hình:** train **mô hình thầy ConvNeXt-V2-Tiny** rồi **chưng cất (knowledge distillation)** sang **mô hình trò MobileNetV4-Conv-Medium** chạy nhanh trên Pi 4. Mô hình có 2 đầu ra: OK/Lỗi và loại lỗi.
6. **Hiệu chỉnh xác suất**, chọn ngưỡng đạt **recall lớp Lỗi ≥ 95 %**, đánh giá riêng trên ảnh tự chụp, Grad-CAM.
7. **Xuất ONNX FP32 + INT8**, đối chiếu sai số với PyTorch, đo tốc độ, sinh `model_card.json` cho `OnnxClassifier` trên Pi.

**Cách chạy trên Google Colab:** Runtime → Change runtime type → **T4 GPU** → điền khóa API ở mục 1 → Runtime → Run all.

### Vì sao chọn mô hình này

| Quyết định | Lý do |
|---|---|
| Phân loại ảnh, không dùng object detection | Camera cố định, mỗi lần chỉ một hộp trong ROI → phân loại cần ít dữ liệu hơn, nhanh hơn và chính xác hơn |
| Thầy: ConvNeXt-V2-Tiny (pretrain FCMAE + ImageNet-22k) | Đặc trưng bề mặt, nếp gấp, vết rách rất tốt; fine-tune ổn định khi dữ liệu ít |
| Trò: MobileNetV4-Conv-Medium | Kiến trúc thiết kế cho CPU di động, chính xác cao trong nhóm dưới 10 triệu tham số |
| Knowledge distillation | Trò học "đáp án mềm" của thầy nên gần bằng độ chính xác của thầy, với tốc độ của trò |
| 2 đầu ra | Đầu 1 OK/Lỗi để quyết định gạt; đầu 2 loại lỗi đa nhãn (rách, móp, bẹp/vỡ, ướt) để hiện lên dashboard |
| Nhúng chuẩn hóa vào mô hình | Trên Pi chỉ cần resize + chia 255, không lo lệch mean/std |
| ONNX + INT8 | Khớp `OnnxClassifier` của Pi; INT8 chỉ được chọn nếu độ chính xác không giảm |

> **Quan trọng:** ảnh công khai và ảnh web khác góc chụp, đèn, nền so với trạm của nhóm. Điểm số dùng cho báo cáo phải lấy trên ảnh tự chụp (`test_own`). Notebook tự tăng trọng số ảnh tự chụp ×3 khi train và không bao giờ đưa ảnh web vào tập test.

> **Bản quyền:** dataset Roboflow phần lớn theo CC BY 4.0 (ghi nguồn trong báo cáo); ảnh cào web chỉ dùng nội bộ cho nghiên cứu và mặc định chỉ lấy ảnh có giấy phép Creative Commons.

## 0. Cài thư viện

In [ ]:
import os, sys, subprocess
SMOKE = os.environ.get("SORTER_SMOKE", "0") == "1"   # chỉ để chạy thử nhanh pipeline — KHÔNG bật khi train thật
IN_COLAB = "google.colab" in sys.modules
os.environ.setdefault("NO_ALBUMENTATIONS_UPDATE", "1")

PKGS = ["timm>=1.0.15", "albumentations>=2.0.5", "onnx", "onnxscript", "onnxruntime>=1.18",
        "open_clip_torch", "datasets", "huggingface_hub", "kagglehub", "kaggle", "icrawler", "ddgs", "roboflow"]
if not SMOKE:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
print("Python", sys.version.split()[0], "| Colab:", IN_COLAB, "| SMOKE:", SMOKE)

## 1. Cấu hình

Khóa API đặt trong **Colab → biểu tượng 🔑 Secrets** (bật "Notebook access"), không dán thẳng vào code:

| Tên secret | Lấy ở đâu | Bắt buộc? |
|---|---|---|
| `ROBOFLOW_API_KEY` | app.roboflow.com → Settings → API Keys (miễn phí) | Nên có |
| `KAGGLE_USERNAME`, `KAGGLE_KEY` | kaggle.com → Settings → API → Create New Token | Tùy chọn |
| `HF_TOKEN` | huggingface.co → Settings → Access Tokens | Tùy chọn |

**Ảnh tự chụp** (từ `capture_dataset.py` trên Pi) để trong Drive theo cấu trúc `<lớp>/<box_id>_<thời gian>.jpg`. Tên thư mục lớp hiểu được: `ok`, `defect`, `tear`/`rach`, `dent`/`mop`, `crush`/`bep`, `wet`/`uot`. Phần `box_id` trước dấu `_` dùng để giữ mọi ảnh của cùng một hộp trong cùng một tập.

In [ ]:
from pathlib import Path

CFG = {
    # ---------- Đường dẫn ----------
    "work_dir": "/content/packaging_ai" if IN_COLAB else str(Path.cwd() / "packaging_ai"),
    "use_drive": IN_COLAB,
    "drive_dir": "/content/drive/MyDrive/smart_sorter/packaging_ai",
    "seed": 42,

    # ---------- Hệ nhãn chung ----------
    "damage_types": ["tear", "dent", "crush", "wet"],          # rách, móp, bẹp/vỡ, ướt-bẩn

    # ---------- Nguồn 1: ảnh tự chụp tại trạm (QUAN TRỌNG NHẤT) ----------
    "own_data_dirs": ["/content/drive/MyDrive/smart_sorter/dataset"],
    "own_data_zips": [],                                       # hoặc file .zip cùng cấu trúc

    # ---------- Nguồn 2: Roboflow Universe ----------
    # mode: "classification" (export dạng thư mục lớp) | "detection" (cắt từng hộp theo bbox) | "image_level" (nhãn cả ảnh từ các vùng lỗi)
    # label_overrides: ép nhãn cho tên lớp lạ, ví dụ {"Open_package": (1, ["tear"])}
    # empty_label=0 (chỉ image_level): ảnh không có vùng lỗi nào được coi là hộp lành
    "use_roboflow": True,
    "roboflow_projects": [
        dict(workspace="aadhavs-first-workspace", project="damaged-vs-good-packages", mode="classification"),
        dict(workspace="ameesh", project="damaged-package-detection-xjkol", mode="classification"),
        dict(workspace="dd-aulv7", project="damaged-package-detection-zoc0p", mode="classification"),
        dict(workspace="project-33xgh", project="damaged-box-auh4b", mode="classification"),
        dict(workspace="nani-tmzf6", project="package-detection-5ozpr", mode="detection"),
        dict(workspace="roboflow-ngkro", project="package-detection-e1ssd", mode="detection"),
        dict(workspace="smart-damage-detection-for-logistics-packages-using-computer-vision",
             project="damaged-boxes-detection-imbqp", mode="image_level"),   # nhãn tear / leakage / squeeze
    ],

    # ---------- Nguồn 3: Kaggle ----------
    "use_kaggle": True,
    "kaggle_datasets": [],                                     # thêm "owner/slug" sau khi xem bảng kết quả tìm kiếm
    "kaggle_search_terms": ["damaged package", "damaged box", "package damage", "cardboard box defect", "parcel damage"],
    "kaggle_auto_download": False,                             # True: tự tải các kết quả có từ khóa lỗi trong tên

    # ---------- Nguồn 4: Hugging Face Hub ----------
    "use_hf": True,
    "hf_search_terms": ["damaged package", "damaged box", "package damage", "cardboard"],
    "hf_datasets": [],                                         # dict(repo="owner/name", split="train", image_col=None, label_col=None)
    "hf_auto_ingest": False,
    "hf_max_per_dataset": 2000,

    # ---------- Nguồn 5: cào ảnh web ----------
    "use_web": True,
    "web_engines": ["bing", "ddg"],
    "web_max_per_query": 60,
    "web_license": "creativecommons",                          # None = lấy mọi giấy phép (không khuyến nghị)
    "web_queries": [
        dict(q="intact cardboard box top view", label=0, types=[]),
        dict(q="new brown shipping box", label=0, types=[]),
        dict(q="sealed carton box on conveyor belt", label=0, types=[]),
        dict(q="thùng carton mới", label=0, types=[]),
        dict(q="torn cardboard box", label=1, types=["tear"]),
        dict(q="ripped shipping package", label=1, types=["tear"]),
        dict(q="thùng carton bị rách", label=1, types=["tear"]),
        dict(q="dented cardboard box", label=1, types=["dent"]),
        dict(q="dented corner shipping box", label=1, types=["dent"]),
        dict(q="hộp carton bị móp", label=1, types=["dent"]),
        dict(q="crushed cardboard box", label=1, types=["crush"]),
        dict(q="crushed parcel delivery damage", label=1, types=["crush"]),
        dict(q="wet cardboard box water damage", label=1, types=["wet"]),
        dict(q="stained dirty shipping box", label=1, types=["wet"]),
        dict(q="damaged parcel box", label=1, types=None),
    ],

    # ---------- Làm sạch & chuẩn hóa ----------
    "min_side": 96, "max_aspect": 4.0, "store_long_side": 512, "phash_max_dist": 4,
    "use_clip_filter": True, "clip_domains": ["web"],
    "clip_arch": "ViT-B-32", "clip_pretrained": "laion2b_s34b_b79k",
    "clip_box_min": 0.55,          # xác suất "là ảnh thùng/hộp" tối thiểu
    "clip_state_min": 0.30,        # xác suất tối thiểu khớp nhãn lành/lỗi

    # ---------- Chia tập & lấy mẫu ----------
    "val_frac": 0.15, "test_frac": 0.15,
    "domain_weight": {"own": 3.0, "public": 1.0, "web": 0.5},
    "synth_defect_prob": 0.15,     # xác suất biến một ảnh lành thành ảnh lỗi tổng hợp khi train

    # ---------- Mô hình & huấn luyện ----------
    "teacher_name": "convnextv2_tiny.fcmae_ft_in22k_in1k",
    "student_name": "mobilenetv4_conv_medium.e500_r256_in1k",
    "use_teacher": True, "pretrained": True,
    "img_size": 256, "batch_size": 48, "num_workers": 2,
    "teacher_epochs": 20, "student_epochs": 30,
    "teacher_lr": 1e-4, "student_lr": 4e-4, "head_lr_mult": 10,
    "weight_decay": 0.05, "warmup_epochs": 1, "label_smoothing": 0.05,
    "type_loss_weight": 0.5, "kd_alpha": 0.5, "kd_temperature": 4.0,
    "patience": 6, "ema_decay": 0.999,

    # ---------- Triển khai ----------
    "target_recall": 0.95, "onnx_opset": 17, "model_version": "1.0.0",
    "run_zoo_bench": True,
    "zoo_models": ["mobilenetv4_conv_small", "mobilenetv4_conv_medium", "mobilenetv3_large_100",
                   "efficientnet_b0", "convnextv2_tiny"],
}

if SMOKE:   # cấu hình chạy thử: dữ liệu giả, mô hình nhỏ, 2 epoch, không mạng
    CFG.update(use_drive=False, use_roboflow=False, use_kaggle=False, use_hf=False, use_web=False,
               use_clip_filter=False, own_data_dirs=[str(Path(CFG["work_dir"]) / "smoke_own")],
               teacher_name="resnet10t", student_name="mobilenetv4_conv_small", pretrained=False,
               img_size=96, batch_size=16, num_workers=0, teacher_epochs=2, student_epochs=2, patience=2,
               ema_decay=0.9, zoo_models=["mobilenetv4_conv_small", "resnet10t"])
print("Đã nạp cấu hình. Mô hình thầy:", CFG["teacher_name"], "| trò:", CFG["student_name"])

In [ ]:
import json, math, random, time, re, shutil, hashlib, zipfile, logging, warnings, unicodedata, io, copy, datetime
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k): return x
try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)
Image.MAX_IMAGE_PIXELS = 80_000_000
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
log = logging.getLogger("pack-ai")

if CFG["use_drive"] and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

WORK = Path(CFG["work_dir"])
RAW, CLEAN, REVIEW = WORK / "raw", WORK / "clean", WORK / "review_web"
OUT, FIG, CKPT = WORK / "export", WORK / "figures", WORK / "checkpoints"
for d in (RAW, CLEAN, REVIEW, OUT, FIG, CKPT):
    d.mkdir(parents=True, exist_ok=True)

random.seed(CFG["seed"]); np.random.seed(CFG["seed"])
TYPES = list(CFG["damage_types"]); NT = len(TYPES)
TYPE_VI = {"tear": "rách", "dent": "móp", "crush": "bẹp/vỡ", "wet": "ướt/bẩn"}
DOMAINS = ["own", "public", "web"]; DOM2ID = {d: i for i, d in enumerate(DOMAINS)}

def get_secret(name):
    """Đọc khóa từ biến môi trường hoặc Colab Secrets; không in ra màn hình."""
    v = os.environ.get(name)
    if v:
        return v
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return None

def show_fig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=120, bbox_inches="tight")
    plt.show(); plt.close(fig)

print("Thư mục làm việc:", WORK)

## 2. Thu thập dữ liệu

### 2.0 Hệ nhãn chung
Mỗi nguồn đặt tên lớp khác nhau (`Box_broken`, `damaged`, `squeeze`, `leakage`, `good`, `thùng carton bị rách`…). Hàm `map_label` đưa mọi tên về 2 thông tin:
- `label`: 0 = OK, 1 = Lỗi, `None` = không rõ (bỏ qua);
- `types`: danh sách loại lỗi đã biết (`[]` với ảnh OK, `None` nếu là lỗi nhưng không rõ loại — đầu ra loại lỗi sẽ không học từ ảnh đó).

Bảng ánh xạ của từng nguồn được in ra để bạn kiểm tra; tên lớp hiểu sai thì sửa bằng `label_overrides` trong cấu hình.

In [ ]:
def _strip_accents(s):
    s = unicodedata.normalize("NFD", str(s)).replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

def _tokens(name):
    s = _strip_accents(name)
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", s).lower()
    return [t for t in re.split(r"[^a-z0-9]+", s) if t]

def _hit(kw, toks, joined):
    # từ khóa ngắn (<= 3 ký tự) phải khớp nguyên token để tránh "ok" khớp nhầm "broken"
    return (kw in toks) if len(kw) <= 3 else (kw in joined)

OK_STRONG = ["undamag", "notdamag", "nodamag", "nondefect", "nodefect", "notdefect", "intact", "good",
             "normal", "perfect", "nguyenven", "tot", "pass"]
DEFECT_KW = ["damag", "broken", "break", "crush", "dent", "tear", "torn", "ripped", "rip", "hole", "wet", "leak",
             "stain", "dirty", "squeez", "deform", "defect", "fault", "burst", "collaps", "bad", "def", "ng", "nok",
             "rach", "mop", "bep", "uot", "hong", "loi"]
OK_KW = ["ok", "box", "boxes", "package", "parcel", "carton", "cardboard", "new", "thung", "hop"]
_TYPE_KW_ALL = {
    "tear":  ["tear", "torn", "ripped", "rip", "hole", "cut", "open", "rach"],
    "dent":  ["dent", "squeez", "deform", "bent", "mop", "lom"],
    "crush": ["crush", "broken", "break", "collaps", "flatten", "burst", "smash", "bep", "vo"],
    "wet":   ["wet", "leak", "stain", "water", "dirty", "mold", "mould", "moist", "uot", "ban"],
}
TYPE_KW = {t: kws for t, kws in _TYPE_KW_ALL.items() if t in TYPES}

def map_label(name):
    """Tên lớp/thư mục/category -> (label, types)."""
    toks = _tokens(name)
    joined = "".join(toks)
    if not toks:
        return None, None
    if any(_hit(k, toks, joined) for k in OK_STRONG):
        return 0, []
    types = [t for t, kws in TYPE_KW.items() if any(_hit(k, toks, joined) for k in kws)]
    if types or any(_hit(k, toks, joined) for k in DEFECT_KW):
        return 1, (types or None)
    if any(_hit(k, toks, joined) for k in OK_KW):
        return 0, []
    return None, None

def merge_labels(pairs):
    """Gộp nhiều (label, types) của cùng một ảnh: có lỗi thì là lỗi, loại lỗi lấy hợp."""
    pairs = [p for p in pairs if p[0] is not None]
    if not pairs:
        return None, None
    d = [t for l, t in pairs if l == 1]
    if not d:
        return 0, []
    known = sorted({x for t in d if t for x in t})
    return 1, (known or None)

RECORDS = []
def add_record(path, source, domain, label, types, group, **meta):
    RECORDS.append(dict(src=str(path), source=source, domain=domain, label=int(label),
                        types=None if types is None else list(types), group=str(group), **meta))

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
def list_images(root):
    return sorted(p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)

def rf_base(stem):
    """Roboflow đặt tên bản augment dạng <gốc>_jpg.rf.<hash>; cắt phần đuôi để gom nhóm."""
    return re.split(r"\.rf\.", stem)[0]

for s in ["ok", "defect", "Box", "Box_broken", "Open_package", "squeeze", "leakage", "undamaged", "thùng carton bị rách", "hộp bị móp"]:
    print(f"{s!r:28} -> {map_label(s)}")

### 2.1 Nguồn 1 — ảnh tự chụp tại trạm

In [ ]:
def ingest_own():
    roots = [Path(p) for p in CFG["own_data_dirs"]]
    for i, z in enumerate(CFG["own_data_zips"]):
        if Path(z).exists():
            dst = RAW / f"own_zip_{i}"
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dst)
            roots.append(dst)
    n, skipped = 0, Counter()
    for root in roots:
        if not root.exists():
            log.warning("Không thấy thư mục ảnh tự chụp: %s", root)
            continue
        for p in list_images(root):
            rel = p.relative_to(root)
            lab, types = None, None
            for part in reversed(rel.parts[:-1]):          # thư mục gần ảnh nhất có tên hiểu được
                lab, types = map_label(part)
                if lab is not None:
                    break
            if lab is None:
                skipped[str(rel.parent)] += 1
                continue
            box_id = re.split(r"[_\-\s]", p.stem)[0]
            add_record(p, "own", "own", lab, types, f"own:{box_id}")
            n += 1
    if skipped:
        log.warning("Bỏ qua ảnh trong thư mục không rõ lớp: %s", dict(skipped))
    print(f"Ảnh tự chụp: {n}")
    return n

### 2.2 Nguồn 2 — Roboflow Universe

Tải phiên bản mới nhất của từng project. Project phân loại xuất dạng thư mục lớp (`folder`), project detection/segmentation xuất dạng COCO rồi:
- `detection`: **cắt từng hộp theo bbox**; vùng lỗi nằm gọn trong một hộp thì gán lỗi cho hộp đó (không cắt riêng vùng lỗi);
- `image_level`: giữ nguyên ảnh, nhãn = gộp nhãn các vùng trong ảnh.

In [ ]:
ROBOFLOW_KEY = get_secret("ROBOFLOW_API_KEY")

def _rf_versions(project, ws, proj):
    nums = []
    try:
        for v in project.versions():
            m = re.search(r"(\d+)$", str(getattr(v, "version", "") or getattr(v, "id", "")))
            if m:
                nums.append(int(m.group(1)))
    except Exception:
        pass
    if not nums:
        try:
            import requests
            js = requests.get(f"https://api.roboflow.com/{ws}/{proj}", params={"api_key": ROBOFLOW_KEY}, timeout=30).json()
            for v in js.get("versions", []):
                tail = str(v.get("id", "")).rsplit("/", 1)[-1]
                if tail.isdigit():
                    nums.append(int(tail))
        except Exception:
            pass
    return sorted(set(nums), reverse=True) or list(range(10, 0, -1))

def download_roboflow(spec):
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_KEY)
    ws, proj = spec["workspace"], spec["project"]
    project = rf.workspace(ws).project(proj)
    fmt = "folder" if spec["mode"] == "classification" else "coco"
    dest = RAW / "roboflow" / f"{ws}__{proj}"
    last = None
    for v in ([spec["version"]] if spec.get("version") else _rf_versions(project, ws, proj)):
        try:
            ds = project.version(v).download(fmt, location=str(dest), overwrite=True)
            return Path(getattr(ds, "location", dest)), v
        except Exception as e:
            last = e
    raise RuntimeError(f"Không tải được {ws}/{proj}: {last}")

def _lab_fn(overrides):
    ov = {k.lower(): tuple(v) for k, v in (overrides or {}).items()}
    return lambda name: ov.get(str(name).lower(), map_label(name))

def ingest_folder_export(root, source, overrides=None):
    """Export phân loại: <split>/<lớp>/*.jpg, hoặc <split>/_classes.csv (đa nhãn)."""
    lab_of = _lab_fn(overrides)
    n, table = 0, {}
    for split_dir in sorted(d for d in Path(root).iterdir() if d.is_dir()):
        csvf = split_dir / "_classes.csv"
        if csvf.exists():
            dfc = pd.read_csv(csvf)
            dfc.columns = [c.strip() for c in dfc.columns]
            fcol = next(c for c in dfc.columns if c.lower() == "filename")
            cls_cols = [c for c in dfc.columns if c != fcol]
            for c in cls_cols:
                table[c] = lab_of(c)
            for _, r in dfc.iterrows():
                p = split_dir / str(r[fcol])
                lab, types = merge_labels([table[c] for c in cls_cols if int(r[c]) == 1])
                if lab is None or not p.exists():
                    continue
                add_record(p, source, "public", lab, types, f"{source}:{rf_base(p.stem)}")
                n += 1
        else:
            for cdir in sorted(d for d in split_dir.iterdir() if d.is_dir()):
                lab, types = lab_of(cdir.name)
                table[cdir.name] = (lab, types)
                if lab is None:
                    continue
                for p in list_images(cdir):
                    add_record(p, source, "public", lab, types, f"{source}:{rf_base(p.stem)}")
                    n += 1
    return n, table

def _area(b): return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
def _inter(a, b):
    return max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))

def ingest_coco_export(root, source, mode, overrides=None, pad=0.10, empty_label=None):
    """empty_label: nhãn cho ảnh không có vùng nào được vẽ (chỉ dùng với image_level). None = bỏ qua ảnh đó;
    0 = coi là hộp lành — chỉ đặt 0 khi chắc dataset vẽ đủ mọi vùng lỗi."""
    lab_of = _lab_fn(overrides)
    crop_dir = RAW / "crops" / re.sub(r"[^A-Za-z0-9_.-]+", "_", source)
    crop_dir.mkdir(parents=True, exist_ok=True)
    n, table = 0, {}
    for ann_file in sorted(Path(root).rglob("_annotations.coco.json")):
        js = json.loads(ann_file.read_text())
        split_dir = ann_file.parent
        cats = {c["id"]: c for c in js.get("categories", [])}
        # Roboflow tự thêm một category gốc (supercategory "none"); bỏ nó nếu còn category khác
        real = {i: c for i, c in cats.items() if str(c.get("supercategory", "")).lower() != "none"} or cats
        cmap = {}
        for i, c in real.items():
            cmap[i] = lab_of(c["name"])
            table[c["name"]] = cmap[i]
        by_img = defaultdict(list)
        for a in js.get("annotations", []):
            lt = cmap.get(a.get("category_id"))
            if lt and lt[0] is not None and a.get("bbox"):
                x, y, w, h = a["bbox"]
                by_img[a["image_id"]].append([x, y, x + w, y + h, lt[0], lt[1]])
        for im in js.get("images", []):
            p = split_dir / im["file_name"]
            anns = by_img.get(im["id"], [])
            gid = f"{source}:{rf_base(Path(im['file_name']).stem)}"
            if not p.exists():
                continue
            if not anns:
                if mode == "image_level" and empty_label == 0:
                    add_record(p, source, "public", 0, [], gid)
                    n += 1
                continue
            if mode == "image_level":
                lab, types = merge_labels([(a[4], a[5]) for a in anns])
                add_record(p, source, "public", lab, types, gid)
                n += 1
                continue
            keep = [True] * len(anns)
            labs = [(a[4], a[5]) for a in anns]
            for i, bi in enumerate(anns):           # vùng nằm gọn trong vật thể lớn hơn -> gộp nhãn vào vật thể đó
                for j, bj in enumerate(anns):
                    if i != j and _area(bj) > 1.5 * _area(bi) and _inter(bi, bj) >= 0.8 * max(_area(bi), 1e-6):
                        keep[i] = False
                        if bi[4] == 1:
                            labs[j] = merge_labels([labs[j], (1, bi[5])])
                        break
            with Image.open(p) as img0:
                img0 = ImageOps.exif_transpose(img0).convert("RGB")
                W, H = img0.size
                for k, b in enumerate(anns):
                    if not keep[k]:
                        continue
                    pw, ph = (b[2] - b[0]) * pad, (b[3] - b[1]) * pad
                    box = (max(0, int(b[0] - pw)), max(0, int(b[1] - ph)), min(W, int(b[2] + pw)), min(H, int(b[3] + ph)))
                    if min(box[2] - box[0], box[3] - box[1]) < CFG["min_side"]:
                        continue
                    out = crop_dir / f"{split_dir.name}_{im['id']}_{k}.jpg"
                    img0.crop(box).save(out, quality=95)
                    add_record(out, source, "public", labs[k][0], labs[k][1], gid)
                    n += 1
    return n, table

def collect_roboflow():
    if not ROBOFLOW_KEY:
        log.warning("Chưa có ROBOFLOW_API_KEY — bỏ qua Roboflow.")
        return
    rows = []
    for spec in CFG["roboflow_projects"]:
        src = f"rf:{spec['project']}"
        try:
            root, ver = download_roboflow(spec)
            if spec["mode"] == "classification":
                n, table = ingest_folder_export(root, src, spec.get("label_overrides"))
            else:
                n, table = ingest_coco_export(root, src, spec["mode"], spec.get("label_overrides"),
                                              empty_label=spec.get("empty_label"))
            rows.append(dict(source=src, version=ver, images=n, label_map=table))
        except Exception as e:
            rows.append(dict(source=src, version=None, images=0, label_map=f"LỖI: {e}"))
    pd.set_option("display.max_colwidth", 200)
    display(pd.DataFrame(rows))

### 2.3 Nguồn 3 — Kaggle
Ô này **tìm kiếm** dataset theo từ khóa và in bảng kết quả. Dataset nào phù hợp thì thêm `owner/slug` vào `CFG["kaggle_datasets"]` rồi chạy lại (hoặc bật `kaggle_auto_download`). Nhãn suy ra từ tên thư mục chứa ảnh.

In [ ]:
def ingest_generic_folder(root, source, domain="public"):
    """Thư mục bất kỳ: nhãn lấy từ thư mục gần ảnh nhất có tên hiểu được, sau đó tới tên file."""
    root = Path(root)
    n, skipped = 0, 0
    for p in list_images(root):
        rel = p.relative_to(root)
        lab, types = None, None
        for part in list(reversed(rel.parts[:-1])) + [p.stem]:
            lab, types = map_label(part)
            if lab is not None:
                break
        if lab is None:
            skipped += 1
            continue
        add_record(p, source, domain, lab, types, f"{source}:{rel.parent}:{rf_base(p.stem)}")
        n += 1
    print(f"{source}: {n} ảnh, bỏ qua {skipped} ảnh không rõ nhãn")
    return n

def _kaggle_ready():
    u, k = get_secret("KAGGLE_USERNAME"), get_secret("KAGGLE_KEY")
    if u and k:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = u, k
        return True
    return Path("~/.kaggle/kaggle.json").expanduser().exists()

def kaggle_search(terms, per_term=8):
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    found = {}
    for t in terms:
        for d in list(api.dataset_list(search=t))[:per_term]:
            ref = str(getattr(d, "ref", d))
            found[ref] = dict(ref=ref, title=getattr(d, "title", ""), term=t)
    return pd.DataFrame(list(found.values()))

def collect_kaggle():
    if not _kaggle_ready():
        log.warning("Chưa có khóa Kaggle — bỏ qua Kaggle.")
        return
    refs = list(CFG["kaggle_datasets"])
    try:
        cand = kaggle_search(CFG["kaggle_search_terms"])
        display(cand)
        if CFG["kaggle_auto_download"] and len(cand):
            refs += [r for r in cand.ref if map_label(r.split("/")[-1])[0] == 1 and r not in refs]
    except Exception as e:
        log.warning("Tìm kiếm Kaggle lỗi: %s", e)
    import kagglehub
    for ref in refs:
        try:
            path = Path(kagglehub.dataset_download(ref))
            ingest_generic_folder(path, f"kaggle:{ref.split('/')[-1]}")
        except Exception as e:
            log.warning("Không tải được Kaggle %s: %s", ref, e)

### 2.4 Nguồn 4 — Hugging Face Hub
Tìm dataset theo từ khóa; chỉ nạp dataset có cột ảnh và cột nhãn dạng `ClassLabel` mà ít nhất một lớp hiểu được là "lỗi".

In [ ]:
def hf_search(terms, limit=10):
    from huggingface_hub import HfApi
    api = HfApi(token=get_secret("HF_TOKEN"))
    rows = []
    for t in terms:
        for d in api.list_datasets(search=t, limit=limit):
            rows.append(dict(id=d.id, term=t, downloads=getattr(d, "downloads", None)))
    return pd.DataFrame(rows).drop_duplicates("id") if rows else pd.DataFrame(columns=["id", "term", "downloads"])

def _hf_features(repo, token):
    from datasets import load_dataset_builder, load_dataset
    try:
        f = load_dataset_builder(repo, token=token).info.features
        if f is not None:
            return f
    except Exception:
        pass
    return load_dataset(repo, split="train", streaming=True, token=token).features

def ingest_hf(repo, split="train", image_col=None, label_col=None, max_n=2000):
    from datasets import load_dataset, Image as HFImage, ClassLabel
    token = get_secret("HF_TOKEN")
    feats = _hf_features(repo, token)
    if feats is None:
        raise ValueError("không đọc được cấu trúc cột")
    image_col = image_col or next((k for k, v in feats.items() if isinstance(v, HFImage)), None)
    label_col = label_col or next((k for k, v in feats.items() if isinstance(v, ClassLabel)), None)
    if not image_col or not label_col:
        raise ValueError(f"thiếu cột ảnh/nhãn: {list(feats)}")
    names = feats[label_col].names
    mp = {i: map_label(nm) for i, nm in enumerate(names)}
    if not any(v[0] == 1 for v in mp.values()):
        raise ValueError(f"không có lớp lỗi trong {names}")
    out = RAW / "hf" / repo.replace("/", "__")
    out.mkdir(parents=True, exist_ok=True)
    src, n = f"hf:{repo.split('/')[-1]}", 0
    for i, ex in enumerate(load_dataset(repo, split=split, streaming=True, token=token)):
        if n >= max_n:
            break
        lab, types = mp.get(ex[label_col], (None, None))
        img = ex[image_col]
        if lab is None or not isinstance(img, Image.Image):
            continue
        f = out / f"{i:06d}.jpg"
        img.convert("RGB").save(f, quality=95)
        add_record(f, src, "public", lab, types, f"{src}:{i}")
        n += 1
    return n, {names[i]: mp[i] for i in mp}

def collect_hf():
    specs = list(CFG["hf_datasets"])
    try:
        cand = hf_search(CFG["hf_search_terms"])
        display(cand.head(30))
        if CFG["hf_auto_ingest"]:
            specs += [dict(repo=r) for r in cand.id if r not in {s["repo"] for s in specs}]
    except Exception as e:
        log.warning("Tìm kiếm Hugging Face lỗi: %s", e)
    for s in specs:
        try:
            n, table = ingest_hf(s["repo"], s.get("split", "train"), s.get("image_col"), s.get("label_col"),
                                 CFG["hf_max_per_dataset"])
            print(f"HF {s['repo']}: {n} ảnh, nhãn {table}")
        except Exception as e:
            log.warning("Bỏ qua HF %s: %s", s["repo"], e)

### 2.5 Nguồn 5 — cào ảnh web (Bing + DuckDuckGo)
Mỗi truy vấn mang sẵn nhãn và loại lỗi. Ảnh web nhiễu nhãn nhiều nhất nên ở mục 3 sẽ được lọc lại bằng CLIP, chỉ dùng cho tập train và có trọng số thấp.

In [ ]:
def _download(url, dest, timeout=12, max_mb=8):
    import requests
    try:
        r = requests.get(url, timeout=timeout, stream=True, headers={"User-Agent": "Mozilla/5.0 (research dataset builder)"})
        if r.status_code != 200 or "image" not in r.headers.get("Content-Type", ""):
            return False
        buf = io.BytesIO()
        for chunk in r.iter_content(65536):
            buf.write(chunk)
            if buf.tell() > max_mb * 2**20:
                return False
        Image.open(io.BytesIO(buf.getvalue())).verify()
        dest.write_bytes(buf.getvalue())
        return True
    except Exception:
        return False

def crawl_bing(q, out_dir, n):
    from icrawler.builtin import BingImageCrawler
    filters = {"license": CFG["web_license"]} if CFG["web_license"] else None
    c = BingImageCrawler(downloader_threads=4, storage={"root_dir": str(out_dir)}, log_level=logging.ERROR)
    c.crawl(keyword=q, max_num=n, filters=filters, file_idx_offset="auto")

def crawl_ddg(q, out_dir, n):
    try:
        from ddgs import DDGS
    except ImportError:
        from duckduckgo_search import DDGS
    kw = {"max_results": n}
    if CFG["web_license"]:
        kw["license_image"] = "Share"
    try:
        res = list(DDGS().images(q, **kw))
    except TypeError:
        kw.pop("license_image", None)
        res = list(DDGS().images(q, **kw))
    urls = [r.get("image") for r in res if r.get("image")]
    out_dir.mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(lambda iu: _download(iu[1], out_dir / f"ddg_{iu[0]:04d}.jpg"), enumerate(urls)))

def collect_web():
    for qi, spec in enumerate(CFG["web_queries"]):
        slug = re.sub(r"[^a-z0-9]+", "_", _strip_accents(spec["q"]).lower()).strip("_")[:40]
        for eng in CFG["web_engines"]:
            out = RAW / "web" / f"{qi:02d}_{slug}" / eng
            out.mkdir(parents=True, exist_ok=True)
            try:
                (crawl_bing if eng == "bing" else crawl_ddg)(spec["q"], out, CFG["web_max_per_query"])
            except Exception as e:
                log.warning("Cào '%s' bằng %s lỗi: %s", spec["q"], eng, e)
            for p in list_images(out):
                add_record(p, f"web_{eng}", "web", spec["label"], spec["types"], f"web:{qi}:{eng}:{p.stem}", query=spec["q"])
        print(f"[{qi + 1}/{len(CFG['web_queries'])}] {spec['q']}")

### 2.6 Chạy thu thập
Ô dưới gọi lần lượt các nguồn đã bật. Nguồn nào lỗi (mất mạng, thiếu khóa, dataset bị xóa) chỉ ghi cảnh báo, các nguồn khác vẫn chạy tiếp.

In [ ]:
def _fake_box(rng, size=320, defect=None):
    """CHỈ dùng cho chế độ chạy thử SMOKE: vẽ một hộp carton giả."""
    img = np.full((size, size, 3), int(rng.integers(70, 120)), np.uint8)
    img = np.clip(img.astype(np.int16) + rng.integers(-12, 12, img.shape), 0, 255).astype(np.uint8)
    w, h = (int(v) for v in rng.integers(int(size * .45), int(size * .8), 2))
    ctr = (size / 2 + float(rng.integers(-20, 20)), size / 2 + float(rng.integers(-20, 20)))
    rect = cv2.boxPoints((ctr, (w, h), float(rng.uniform(-20, 20)))).astype(np.int32)
    col = tuple(int(c) for c in rng.integers([140, 95, 55], [185, 130, 85]))
    cv2.fillPoly(img, [rect], col)
    cx, cy = (int(v) for v in rect.mean(0))
    cv2.rectangle(img, (cx - 28, cy - 28), (cx + 28, cy + 28), (240, 240, 240), -1)
    for _ in range(30):
        x, y = (int(v) for v in rng.integers(-24, 20, 2))
        cv2.rectangle(img, (cx + x, cy + y), (cx + x + 4, cy + y + 4), (20, 20, 20), -1)
    if defect == "tear":
        pts = (np.array([cx, cy]) + rng.integers(-60, 60, (9, 2))).astype(np.int32)
        cv2.fillPoly(img, [cv2.convexHull(pts)], (60, 40, 25))
    elif defect == "dent":
        cv2.ellipse(img, (cx + 40, cy - 30), (35, 22), 30, 0, 360, (95, 65, 40), -1)
    elif defect == "crush":
        for k in range(5):
            p0 = tuple(int(v) for v in rect[0] + rng.integers(0, 50, 2))
            cv2.line(img, p0, (p0[0] + int(rng.integers(20, 60)), p0[1] + int(rng.integers(-30, 30))), (50, 35, 20), 3)
    elif defect == "wet":
        cv2.circle(img, (cx - 35, cy + 35), 40, tuple(int(c * .6) for c in col), -1)
    return img

def _write(img, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), img[:, :, ::-1])

def make_smoke_fixtures():
    """Sinh dữ liệu giả để kiểm tra toàn bộ pipeline mà không cần mạng (chế độ SMOKE)."""
    rng = np.random.default_rng(0)
    own = Path(CFG["own_data_dirs"][0])
    for b in range(48):
        cls = ["ok", "ok", "tear", "dent", "crush", "wet"][b % 6]
        for k in range(3):
            _write(_fake_box(rng, defect=None if cls == "ok" else cls), own / cls / f"B{b:03d}_{k}.jpg")
    fr = RAW / "smoke_rf_folder"                                   # giống export "folder" của Roboflow
    for split in ["train", "valid"]:
        for cls, d in [("good", None), ("damaged", "tear")]:
            for i in range(8):
                img = _fake_box(rng, defect=d)
                for a in range(2):                                 # 2 bản augment của cùng ảnh gốc
                    _write(np.ascontiguousarray(np.rot90(img, a)), fr / split / cls / f"img{split}{cls}{i}_jpg.rf.{a}abc{i}.jpg")
    rows = []
    for i in range(6):                                             # export đa nhãn _classes.csv
        d = "dent" if i % 2 else None
        _write(_fake_box(rng, defect=d), fr / "test" / f"ml{i}_jpg.rf.x{i}.jpg")
        rows.append({"filename": f"ml{i}_jpg.rf.x{i}.jpg", " Box": 0 if d else 1, " Box_broken": 1 if d else 0})
    pd.DataFrame(rows).to_csv(fr / "test" / "_classes.csv", index=False)
    fc = RAW / "smoke_rf_coco" / "train"                           # giống export "coco" của Roboflow
    images, anns, aid = [], [], 0
    for i in range(10):
        canvas = np.full((300, 420, 3), 90, np.uint8)
        canvas[20:200, 20:200] = _fake_box(rng, 180)
        canvas[40:260, 210:400] = cv2.resize(_fake_box(rng, 200, "crush"), (190, 220))
        _write(canvas, fc / f"c{i}_jpg.rf.q{i}.jpg")
        images.append({"id": i, "file_name": f"c{i}_jpg.rf.q{i}.jpg", "width": 420, "height": 300})
        boxes = [(1, [20, 20, 180, 180]), (2, [210, 40, 190, 220])]
        if i % 2 == 0:
            boxes.append((3, [60, 60, 40, 40]))                    # vùng rách nằm trong hộp "Box"
        for cid, bb in boxes:
            anns.append({"id": aid, "image_id": i, "category_id": cid, "bbox": bb}); aid += 1
    cats = [{"id": 0, "name": "packages", "supercategory": "none"}, {"id": 1, "name": "Box", "supercategory": "packages"},
            {"id": 2, "name": "Box_broken", "supercategory": "packages"}, {"id": 3, "name": "tear", "supercategory": "packages"}]
    (fc / "_annotations.coco.json").write_text(json.dumps({"images": images, "annotations": anns, "categories": cats}))
    fk = RAW / "smoke_kaggle" / "data"                             # thư mục kiểu Kaggle
    for cls, d in [("intact", None), ("damaged", "wet")]:
        for i in range(10):
            _write(_fake_box(rng, defect=d), fk / cls / f"{cls}_{i}.png")
    fw = RAW / "smoke_web"                                         # giả ảnh web, kèm 2 ảnh trùng gần đúng với ảnh tự chụp
    for i in range(12):
        _write(_fake_box(rng, defect="tear" if i % 2 else None), fw / ("torn cardboard box" if i % 2 else "new brown box") / f"w{i}.jpg")
    shutil.copy(own / "ok" / "B000_0.jpg", fw / "new brown box" / "dup_of_own.jpg")
    _write(np.zeros((40, 40, 3), np.uint8), fw / "new brown box" / "tiny.jpg")

def smoke_ingest():
    n1, t1 = ingest_folder_export(RAW / "smoke_rf_folder", "rf:smoke-folder")
    n2, t2 = ingest_coco_export(RAW / "smoke_rf_coco", "rf:smoke-coco", "detection")
    n3 = ingest_generic_folder(RAW / "smoke_kaggle", "kaggle:smoke")
    n4 = ingest_generic_folder(RAW / "smoke_web", "web_smoke", domain="web")
    print("folder:", n1, t1, "\ncoco:", n2, t2)
    coco = [r for r in RECORDS if r["source"] == "rf:smoke-coco"]
    assert sum(r["label"] == 1 and r["types"] == ["crush"] for r in coco) == 10, "hộp Box_broken phải là lỗi 'crush'"
    assert sum(r["label"] == 1 and r["types"] == ["tear"] for r in coco) == 5, "vùng rách phải được gộp vào hộp chứa nó"
    assert sum(r["label"] == 0 for r in coco) == 5
    ml = [r for r in RECORDS if r["source"] == "rf:smoke-folder" and "ml" in Path(r["src"]).name]
    assert len(ml) == 6 and sum(r["label"] for r in ml) == 3

if SMOKE:
    make_smoke_fixtures()

RECORDS.clear()
ingest_own()
if CFG["use_roboflow"]: collect_roboflow()
if CFG["use_kaggle"]:   collect_kaggle()
if CFG["use_hf"]:       collect_hf()
if CFG["use_web"]:      collect_web()
if SMOKE:               smoke_ingest()

raw_df = pd.DataFrame(RECORDS)
assert len(raw_df), "Chưa có ảnh nào — kiểm tra đường dẫn ảnh tự chụp và khóa API."
raw_df.to_csv(WORK / "raw_records.csv", index=False)
print("Tổng số ảnh thô:", len(raw_df))
display(raw_df.groupby(["domain", "source", "label"]).size().unstack(fill_value=0))
if (raw_df.domain == "own").sum() == 0:
    log.warning("CHƯA CÓ ẢNH TỰ CHỤP. Mô hình vẫn train được nhưng độ chính xác trên trạm thật sẽ không được kiểm chứng.")

## 3. Chuẩn hóa và làm sạch dữ liệu

Các bước, theo thứ tự:
1. Đọc lại mọi ảnh, xoay theo EXIF, đổi về RGB (ảnh trong suốt được đặt lên nền trắng), bỏ ảnh hỏng, cạnh ngắn < `min_side`, tỉ lệ cạnh quá lệch, ảnh gần như một màu.
2. Thu nhỏ cạnh dài về `store_long_side` và lưu JPEG thống nhất vào `clean/`.
3. **Khử trùng lặp:** trùng tuyệt đối (SHA-1 điểm ảnh) và trùng gần đúng (perceptual hash, khoảng cách Hamming ≤ `phash_max_dist`). Ảnh tự chụp luôn được tin và giữ lại (cùng một hộp chụp trước/sau khi làm rách vẫn giữ cả hai). Ảnh nguồn ngoài trùng với ảnh khác nhãn bị loại vì nhãn đáng ngờ; trùng cùng nhãn thì chỉ giữ một bản. Ảnh trùng nhau luôn được xếp vào cùng một tập khi chia.
4. **Lọc ảnh web bằng CLIP:** bỏ ảnh không phải thùng/hộp và ảnh có nhãn ngược với nội dung; các ảnh này được chép sang `review_web/` để bạn xem lại bằng tay.

In [ ]:
DOMAIN_PRIORITY = {"own": 0, "public": 1, "web": 2}

def phash64(im):
    """Perceptual hash 64 bit (DCT 32x32 -> 8x8), tương đương imagehash.phash."""
    g = np.asarray(im.convert("L").resize((32, 32), Image.LANCZOS), dtype=np.float32)
    d = cv2.dct(g)[:8, :8].flatten()
    bits = d > np.median(d[1:])
    return int("".join("1" if b else "0" for b in bits), 2)

def normalize_one(rec):
    try:
        with Image.open(rec["src"]) as im:
            im = ImageOps.exif_transpose(im)
            if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
                im = im.convert("RGBA")
                bg = Image.new("RGB", im.size, (255, 255, 255))
                bg.paste(im, mask=im.split()[-1])
                im = bg
            else:
                im = im.convert("RGB")
    except Exception:
        return None, "không đọc được"
    w, h = im.size
    if min(w, h) < CFG["min_side"]:
        return None, "quá nhỏ"
    if max(w, h) / min(w, h) > CFG["max_aspect"]:
        return None, "tỉ lệ cạnh bất thường"
    s = CFG["store_long_side"] / max(w, h)
    if s < 1:
        im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.LANCZOS)
    arr = np.asarray(im)
    if arr.std() < 4:
        return None, "gần như một màu"
    sha = hashlib.sha1(arr.tobytes()).hexdigest()
    out = CLEAN / rec["domain"] / f"{sha[:24]}.jpg"
    out.parent.mkdir(parents=True, exist_ok=True)
    if not out.exists():
        im.save(out, quality=92)
    return dict(rec, path=str(out), sha=sha, phash=phash64(im), w=im.size[0], h=im.size[1]), None

def near_dup_pairs(hashes, max_dist):
    """Tìm cặp hash cách nhau <= max_dist bit. Nguyên lý chuồng bồ câu: chia 64 bit thành max_dist+1 dải,
    hai hash gần nhau chắc chắn trùng nguyên ít nhất một dải -> chỉ so sánh trong cùng bucket."""
    nb = max_dist + 1
    bounds = np.linspace(0, 64, nb + 1).astype(int)
    pairs = set()
    for b in range(nb):
        lo, hi = int(bounds[b]), int(bounds[b + 1])
        mask = (1 << (hi - lo)) - 1
        buckets = defaultdict(list)
        for i, h in enumerate(hashes):
            buckets[(h >> lo) & mask].append(i)
        for idx in buckets.values():
            if len(idx) < 2 or len(idx) > 3000:
                continue
            for a in range(len(idx)):
                ha = hashes[idx[a]]
                for c in range(a + 1, len(idx)):
                    if bin(ha ^ hashes[idx[c]]).count("1") <= max_dist:
                        pairs.add((idx[a], idx[c]))
    return pairs

class UnionFind:
    def __init__(self):
        self.p = {}
    def find(self, x):
        self.p.setdefault(x, x)
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra

# ---- 1-2. Chuẩn hóa ----
reasons = Counter()
with ThreadPoolExecutor(8) as ex:
    results = list(tqdm(ex.map(normalize_one, RECORDS), total=len(RECORDS), desc="Chuẩn hóa"))
recs = []
for r, why in results:
    if r is None:
        reasons[why] += 1
    else:
        recs.append(r)
print("Bị loại khi chuẩn hóa:", dict(reasons))

# ---- 3. Khử trùng lặp ----
uf = UnionFind()
first_by_sha = {}
for i, r in enumerate(recs):
    uf.find(i)
    if r["sha"] in first_by_sha:
        uf.union(first_by_sha[r["sha"]], i)
    else:
        first_by_sha[r["sha"]] = i
for i, j in near_dup_pairs([r["phash"] for r in recs], CFG["phash_max_dist"]):
    uf.union(i, j)
clusters = defaultdict(list)
for i in range(len(recs)):
    clusters[uf.find(i)].append(i)

def hdist(a, b):
    return bin(recs[a]["phash"] ^ recs[b]["phash"]).count("1")

def same_img(a, b):
    return recs[a]["sha"] == recs[b]["sha"] or hdist(a, b) <= CFG["phash_max_dist"]

kept_idx, conflicts, dropped_dup = [], [], 0
group_uf = UnionFind()
for root, mem in clusters.items():
    mem = sorted(mem, key=lambda m: (DOMAIN_PRIORITY[recs[m]["domain"]], -recs[m]["w"] * recs[m]["h"]))
    keep_here = []
    for m in mem:
        r = recs[m]
        if r["domain"] == "own":
            # Ảnh tự chụp do nhóm gán nhãn: luôn tin. Cùng một hộp chụp trước/sau khi làm rách có thể rất giống nhau
            # nhưng khác nhãn — vẫn giữ cả hai. Chỉ bỏ bản trùng tuyệt đối cùng nhãn.
            if any(recs[k]["sha"] == r["sha"] and recs[k]["label"] == r["label"] for k in keep_here):
                dropped_dup += 1
                continue
            keep_here.append(m)
            continue
        near = [k for k in mem if k != m and same_img(k, m)]
        if any(recs[k]["label"] != r["label"] for k in near):
            conflicts.append(r["src"])                  # ảnh nguồn ngoài trùng một ảnh khác nhãn -> nhãn đáng ngờ
            continue
        dup_of = next((k for k in keep_here if same_img(k, m)), None)
        if dup_of is not None:                          # trùng ảnh đã giữ (cùng nhãn): bỏ, gộp thông tin loại lỗi
            dropped_dup += 1
            if r["label"] == 1 and r["types"]:
                recs[dup_of]["types"] = sorted(set(recs[dup_of]["types"] or []) | set(r["types"]))
            continue
        keep_here.append(m)
    for m in keep_here:
        group_uf.union(f"g:{recs[m]['group']}", f"c:{root}")
        kept_idx.append(m)
kept = [dict(recs[m], cluster=f"c{uf.find(m)}") for m in kept_idx]
for r in kept:
    r["split_group"] = group_uf.find(f"g:{r['group']}")
print(f"Giữ {len(kept)} ảnh | bỏ {dropped_dup} ảnh trùng | bỏ {len(conflicts)} ảnh nguồn ngoài trùng với ảnh khác nhãn")
if conflicts:
    pd.DataFrame({"file": conflicts}).to_csv(WORK / "label_conflicts.csv", index=False)

df = pd.DataFrame(kept)
df["types_str"] = df["types"].apply(lambda t: "?" if t is None else "|".join(t))
df["type_known"] = df.apply(lambda r: int(r["label"] == 1 and r["types"] is not None and len(r["types"]) > 0), axis=1)
df["phash"] = df["phash"].apply(lambda h: f"{h:016x}")

In [ ]:
# ---- 4. Lọc ảnh web bằng CLIP ----
def clip_scores(paths, bs=64):
    import torch, open_clip
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model, _, preprocess = open_clip.create_model_and_transforms(CFG["clip_arch"], pretrained=CFG["clip_pretrained"], device=dev)
    tok = open_clip.get_tokenizer(CFG["clip_arch"])
    box_p = ["a photo of a cardboard box", "a photo of a shipping package", "a photo of a carton box"]
    other_p = ["a photo of a person", "a drawing or cartoon", "a logo or text", "a screenshot of a website",
               "a photo of a room interior", "a photo of a pile of trash"]
    ok_p = ["a photo of an intact undamaged cardboard box", "a photo of a new clean shipping box"]
    bad_p = ["a photo of a damaged cardboard box", "a photo of a torn, dented or crushed shipping box",
             "a photo of a wet stained cardboard box"]
    with torch.no_grad():
        txt = model.encode_text(tok(box_p + other_p + ok_p + bad_p).to(dev)).float()
        txt /= txt.norm(dim=-1, keepdim=True)
        nb, no, nk = len(box_p), len(other_p), len(ok_p)
        p_box, p_bad = [], []
        for i in tqdm(range(0, len(paths), bs), desc="CLIP"):
            ims = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in paths[i:i + bs]]).to(dev)
            f = model.encode_image(ims).float()
            f /= f.norm(dim=-1, keepdim=True)
            lg = 100.0 * f @ txt.T
            a = lg[:, :nb + no].softmax(1)
            p_box.append(a[:, :nb].sum(1).cpu())
            b = lg[:, nb + no:].softmax(1)
            p_bad.append(b[:, nk:].sum(1).cpu())
    return torch.cat(p_box).numpy(), torch.cat(p_bad).numpy()

df["clip_box"], df["clip_bad"] = np.nan, np.nan
if CFG["use_clip_filter"]:
    m = df.domain.isin(CFG["clip_domains"])
    if m.any():
        pb, pd_ = clip_scores(df.loc[m, "path"].tolist())
        df.loc[m, "clip_box"], df.loc[m, "clip_bad"] = pb, pd_
        state_ok = np.where(df.loc[m, "label"] == 1, pd_, 1 - pd_)
        bad = pd.Series((pb < CFG["clip_box_min"]) | (state_ok < CFG["clip_state_min"]), index=df.index[m.to_numpy()])
        drop_idx = bad[bad].index
        for i in drop_idx:
            reason = "khong_phai_hop" if df.at[i, "clip_box"] < CFG["clip_box_min"] else "nhan_nghi_sai"
            dst = REVIEW / reason / Path(df.at[i, "path"]).name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(df.at[i, "path"], dst)
        df = df.drop(index=drop_idx).reset_index(drop=True)
        print(f"CLIP loại {len(drop_idx)} ảnh web (đã chép sang {REVIEW} để xem lại)")

df.to_csv(WORK / "manifest_clean.csv", index=False)
print("Số ảnh sau làm sạch:", len(df))

In [ ]:
# ---- Thống kê sau làm sạch ----
display(pd.crosstab([df.domain, df.source], df.label.map({0: "OK", 1: "Lỗi"}), margins=True))
type_counts = Counter(t for ts in df.loc[df.label == 1, "types"] if ts for t in ts)
print("Loại lỗi đã biết:", {f"{t} ({TYPE_VI.get(t, t)})": type_counts.get(t, 0) for t in TYPES},
      "| ảnh lỗi chưa rõ loại:", int(((df.label == 1) & (df.type_known == 0)).sum()))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
pd.crosstab(df.domain, df.label.map({0: "OK", 1: "Lỗi"})).plot.bar(ax=ax[0], rot=0, title="Số ảnh theo nguồn và nhãn")
pd.Series({TYPE_VI.get(t, t): type_counts.get(t, 0) for t in TYPES}).plot.bar(ax=ax[1], rot=0, title="Số ảnh theo loại lỗi")
show_fig(fig, "data_stats")

def show_grid(paths, titles, ncol=8, name="grid", size=2.0):
    n = len(paths)
    if n == 0:
        return
    nrow = math.ceil(n / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * size, nrow * size * 1.1), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, p, t in zip(axes.flat, paths, titles):
        img = p if isinstance(p, np.ndarray) else np.asarray(Image.open(p).convert("RGB"))
        ax.imshow(img); ax.set_title(t, fontsize=7)
    show_fig(fig, name)

smp = pd.concat([g.sample(min(len(g), 4), random_state=0) for _, g in df.groupby(["domain", "label"])])
show_grid(smp.path.tolist(), [f"{d} | {'Lỗi' if l else 'OK'} {t if l else ''}" for d, l, t in zip(smp.domain, smp.label, smp.types_str)],
          name="samples")

## 4. Chia tập train / val / test theo nhóm

- Ảnh tự chụp và ảnh công khai được chia **riêng từng miền**, mỗi miền 70/15/15, theo `split_group` (cùng hộp, cùng ảnh gốc, hoặc trùng gần đúng → cùng một tập), giữ tỉ lệ OK/Lỗi.
- Ảnh web **chỉ vào train** vì nhãn nhiễu.
- Ô cuối kiểm tra không có nhóm nào nằm ở hai tập.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

def split_groups(sub, frac, seed):
    """Tách sub thành (phần còn lại, phần chiếm khoảng frac) theo nhóm, cố giữ tỉ lệ nhãn."""
    if len(sub) == 0 or frac <= 0:
        return sub.index, sub.index[:0]
    k = max(2, int(round(1 / frac)))
    n_groups = sub.split_group.nunique()
    if n_groups < 2:
        return sub.index, sub.index[:0]
    try:
        if n_groups < 2 * k or sub.label.nunique() < 2:
            raise ValueError("ít nhóm")
        a, b = next(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
                    .split(sub, sub.label, sub.split_group))
    except ValueError:
        a, b = next(GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed).split(sub, groups=sub.split_group))
    return sub.index[a], sub.index[b]

df["split"] = "train"
for dom in ["own", "public"]:
    sub = df[df.domain == dom]
    rest_idx, test_idx = split_groups(sub, CFG["test_frac"], CFG["seed"])
    rest = df.loc[rest_idx]
    _, val_idx = split_groups(rest, CFG["val_frac"] / (1 - CFG["test_frac"]), CFG["seed"] + 1)
    df.loc[test_idx, "split"] = "test"
    df.loc[val_idx, "split"] = "val"

leak = df.groupby("split_group").split.nunique()
assert (leak <= 1).all(), "Có nhóm ảnh nằm ở nhiều tập!"
df.to_csv(WORK / "manifest_split.csv", index=False)
display(pd.crosstab([df.split, df.domain], df.label.map({0: "OK", 1: "Lỗi"}), margins=True))

n_own_test = int(((df.split == "test") & (df.domain == "own")).sum())
if n_own_test < 40:
    log.warning("Tập test ảnh tự chụp chỉ có %d ảnh — nên chụp thêm (mục tiêu ≥ 300 ảnh mỗi lớp).", n_own_test)

## 5. Tăng cường dữ liệu

### 5.1 Sinh lỗi tổng hợp
Ảnh lỗi thật thường ít hơn ảnh lành. Khi train, mỗi ảnh lành có xác suất `synth_defect_prob` được vẽ thêm một loại lỗi:
- **rách:** vùng đa giác răng cưa lộ lớp giấy sóng tối màu, viền sợi giấy sáng;
- **móp:** biến dạng lõm hướng tâm + bóng đổ một phía + nếp gấp;
- **bẹp/vỡ:** một vùng gần mép thân hộp bị ép lún về phía tâm + nhiều nếp gấp toả ra;
- **ướt/bẩn:** vệt loang tối màu, viền đậm như nước khô.

Lỗi được đặt ở vùng giữa ảnh vì ảnh đầu vào là ROI đã cắt quanh hộp (thân hộp chiếm phần lớn khung). Ảnh tổng hợp chỉ xuất hiện trong train, không bao giờ trong val/test — nên điểm đánh giá luôn là điểm trên ảnh thật.

In [ ]:
def _soft(mask, k):
    k = int(k) | 1
    return cv2.GaussianBlur(mask.astype(np.float32), (k, k), 0)

def _jagged_poly(cx, cy, r, n, rng, jitter=(0.45, 1.35)):
    ang = np.sort(rng.uniform(0, 2 * np.pi, n))
    rad = r * rng.uniform(*jitter, n)
    return np.stack([cx + rad * np.cos(ang), cy + rad * np.sin(ang)], 1).astype(np.int32)

def synth_tear(img, rng):
    h, w = img.shape[:2]; m = min(h, w)
    cx, cy = rng.uniform(0.3, 0.7) * w, rng.uniform(0.3, 0.7) * h      # vùng giữa ROI = thân hộp
    mask = np.zeros((h, w), np.uint8)
    cv2.fillPoly(mask, [_jagged_poly(cx, cy, rng.uniform(0.05, 0.16) * m, int(rng.integers(14, 28)), rng)], 1)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    th = rng.uniform(0, np.pi)
    wave = 0.5 + 0.5 * np.sin((xx * np.cos(th) + yy * np.sin(th)) * 2 * np.pi / max(3.0, m * 0.02))
    r0 = rng.uniform(70, 130)
    base = np.float32([r0, r0 * rng.uniform(0.65, 0.8), r0 * rng.uniform(0.35, 0.5)])   # nâu giấy sóng
    inner = base * (0.55 + 0.35 * wave[..., None])
    edge = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=int(rng.integers(1, 3))) - mask
    a = _soft(mask, 3)[..., None]
    out = img.astype(np.float32) * (1 - a) + inner * a
    e = edge.astype(bool)
    out[e] = out[e] * 0.4 + np.array([235, 225, 205], np.float32) * 0.6
    return np.clip(out, 0, 255).astype(np.uint8)

def synth_dent(img, rng):
    h, w = img.shape[:2]; m = min(h, w)
    cx, cy, r = rng.uniform(0.3, 0.7) * w, rng.uniform(0.3, 0.7) * h, rng.uniform(0.08, 0.2) * m
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    d = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2) / r
    f = np.where(d < 1, 1 - rng.uniform(0.15, 0.35) * (1 - d ** 2), 1.0).astype(np.float32)
    out = cv2.remap(img, (cx + (xx - cx) * f).astype(np.float32), (cy + (yy - cy) * f).astype(np.float32),
                    cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    ang = rng.uniform(0, 2 * np.pi)
    proj = ((xx - cx) * np.cos(ang) + (yy - cy) * np.sin(ang)) / r
    shade = np.clip(np.exp(-1.5 * d ** 2) * 0.5 * proj, -0.45, 0.3)
    out = out.astype(np.float32) * (1 + shade[..., None])
    if rng.random() < 0.75:
        crease = np.zeros((h, w), np.float32)
        p0 = np.array([cx, cy]) + rng.uniform(-r, r, 2)
        p1 = np.array([cx, cy]) + rng.uniform(-r, r, 2)
        cv2.line(crease, tuple(int(v) for v in p0), tuple(int(v) for v in p1), 1.0, max(1, int(m * 0.006)))
        out *= (1 - 0.35 * _soft(crease, 3))[..., None]
    return np.clip(out, 0, 255).astype(np.uint8)

def synth_crush(img, rng):
    """Bẹp/vỡ: một vùng gần mép thân hộp bị ép lún về phía tâm + nhiều nếp gấp toả ra."""
    h, w = img.shape[:2]; m = min(h, w)
    ang = rng.uniform(0, 2 * np.pi)
    R, rho = rng.uniform(0.15, 0.28) * m, rng.uniform(0.12, 0.22) * m
    px, py = w / 2 + R * np.cos(ang), h / 2 + R * np.sin(ang)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    d = np.sqrt((xx - px) ** 2 + (yy - py) ** 2) / rho
    k = np.where(d < 1, (1 - d ** 2) ** 2, 0).astype(np.float32) * rng.uniform(0.25, 0.45) * rho
    mapx = (xx + k * np.cos(ang)).astype(np.float32)   # lấy mẫu từ phía ngoài -> bề mặt bị đẩy lún vào trong
    mapy = (yy + k * np.sin(ang)).astype(np.float32)
    out = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101).astype(np.float32)
    out *= (1 - 0.25 * np.clip(1 - d, 0, 1))[..., None]                 # vùng lún tối hơn
    crease = np.zeros((h, w), np.float32)
    for _ in range(int(rng.integers(3, 7))):
        a = ang + np.pi + rng.uniform(-0.9, 0.9)
        L = rng.uniform(0.4, 1.0) * rho
        p0 = np.array([px, py]) + rng.uniform(-0.3, 0.3, 2) * rho
        p1 = p0 + L * np.array([np.cos(a), np.sin(a)])
        cv2.line(crease, tuple(int(v) for v in p0), tuple(int(v) for v in p1), 1.0, max(1, int(m * 0.006)))
    crease = _soft(crease, 3)
    hl = np.roll(crease, 2, axis=1)
    out = out * (1 - 0.45 * crease)[..., None] + 60 * (hl * (1 - crease))[..., None]
    return np.clip(out, 0, 255).astype(np.uint8)

def synth_wet(img, rng):
    h, w = img.shape[:2]
    noise = rng.random((max(4, h // 24), max(4, w // 24))).astype(np.float32)
    noise = cv2.GaussianBlur(cv2.resize(noise, (w, h), interpolation=cv2.INTER_CUBIC), (0, 0), min(h, w) * 0.03)
    cx, cy, r = rng.uniform(0.3, 0.7) * w, rng.uniform(0.3, 0.7) * h, rng.uniform(0.1, 0.25) * min(h, w)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    field = noise * 0.5 + np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * r * r)) * 0.8
    mask = _soft((field > np.quantile(field, rng.uniform(0.86, 0.95))).astype(np.float32), 9)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 2] *= 1 - mask * rng.uniform(0.25, 0.45)
    hsv[..., 1] *= 1 + mask * 0.15
    out = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32)
    ring = np.clip(cv2.Laplacian(mask, cv2.CV_32F), 0, None)
    ring /= ring.max() + 1e-6
    out *= (1 - 0.25 * ring)[..., None]
    return np.clip(out, 0, 255).astype(np.uint8)

SYNTH = {k: v for k, v in {"tear": synth_tear, "dent": synth_dent, "crush": synth_crush, "wet": synth_wet}.items() if k in TYPES}

_ok = df[(df.label == 0) & (df.split == "train")].sample(min(4, int(((df.label == 0) & (df.split == "train")).sum())), random_state=1)
_imgs, _tt = [], []
for p in _ok.path:
    base = np.asarray(Image.open(p).convert("RGB"))
    _imgs.append(base); _tt.append("gốc")
    for k, fn in SYNTH.items():
        _imgs.append(fn(base, np.random.default_rng(len(_imgs)))); _tt.append(TYPE_VI.get(k, k))
show_grid(_imgs, _tt, ncol=1 + len(SYNTH), name="synthetic_defects", size=2.4)

### 5.2 Tăng cường bằng albumentations
- **Hình học:** cắt-phóng ngẫu nhiên, lật ngang/dọc, xoay 90° (camera nhìn thẳng xuống nên hướng hộp không quan trọng), xoay nhẹ, nghiêng.
- **Ánh sáng & màu:** độ sáng/tương phản, gamma, CLAHE, sắc độ/bão hòa, đổ bóng, ảnh xám.
- **Cảm biến:** mờ, mờ chuyển động, nhiễu Gauss, nén JPEG.

Không dùng CutMix/che ô ngẫu nhiên vì một vết rách nhỏ đã đủ làm cả hộp bị loại — trộn ảnh sẽ làm sai nhãn. Tập val/test chỉ resize thẳng về `img_size` bằng `INTER_AREA`, **giống hệt cách Pi xử lý ROI**.

In [ ]:
import albumentations as A

S = CFG["img_size"]
train_tf = A.Compose([
    A.RandomResizedCrop(size=(S, S), scale=(0.55, 1.0), ratio=(0.75, 1.333)),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-15, 15), shear=(-5, 5), p=0.5),
    A.OneOf([A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25),
             A.RandomGamma(gamma_limit=(80, 120)), A.CLAHE(clip_limit=2.0)], p=0.7),
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=15, p=0.4),
    A.OneOf([A.GaussianBlur(blur_limit=(3, 5)), A.MotionBlur(blur_limit=(3, 7))], p=0.25),
    A.GaussNoise(std_range=(0.02, 0.08), p=0.25),
    A.ImageCompression(quality_range=(50, 95), p=0.3),
    A.RandomShadow(p=0.15),
    A.ToGray(p=0.05),
])
eval_tf = A.Compose([A.Resize(height=S, width=S, interpolation=cv2.INTER_AREA)])

_tr = df[df.split == "train"]
_p = _tr.sample(min(2, len(_tr)), random_state=3).path.tolist()
_aug = []
for p in _p:
    im = np.asarray(Image.open(p).convert("RGB"))
    _aug += [eval_tf(image=im)["image"]] + [train_tf(image=im)["image"] for _ in range(5)]
show_grid(_aug, (["gốc"] + ["tăng cường"] * 5) * len(_p), ncol=6, name="augmentations")

### 5.3 Dataset và bộ lấy mẫu cân bằng
Bộ lấy mẫu `WeightedRandomSampler` làm hai lớp OK/Lỗi xuất hiện đều nhau trong mỗi epoch; trong cùng một lớp, ảnh tự chụp được bốc nhiều gấp 3 lần ảnh công khai, ảnh web chỉ bằng một nửa.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

torch.manual_seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
if USE_AMP:
    torch.backends.cudnn.benchmark = True
print("Thiết bị:", DEVICE, torch.cuda.get_device_name(0) if USE_AMP else "")

def type_vec(types):
    v = np.zeros(NT, np.float32)
    for t in (types or []):
        if t in TYPES:
            v[TYPES.index(t)] = 1.0
    return v

class PackDS(Dataset):
    def __init__(self, d, tf, train=False):
        self.paths = d.path.tolist()
        self.y = d.label.astype(int).tolist()
        self.tv = np.stack([type_vec(t) for t in d.types]) if len(d) else np.zeros((0, NT), np.float32)
        self.tm = d.type_known.astype(np.float32).tolist()
        self.dom = [DOM2ID[x] for x in d.domain]
        self.tf, self.train = tf, train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = cv2.cvtColor(cv2.imread(self.paths[i], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        y, tv, tm = self.y[i], self.tv[i].copy(), self.tm[i]
        if self.train and y == 0 and SYNTH and random.random() < CFG["synth_defect_prob"]:
            kind = random.choice(list(SYNTH))
            img = SYNTH[kind](img, np.random.default_rng(random.getrandbits(32)))
            y, tv, tm = 1, type_vec([kind]), 1.0
        img = self.tf(image=img)["image"]
        x = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1))).float().div_(255.0)
        return x, torch.tensor(y), torch.from_numpy(tv), torch.tensor(tm, dtype=torch.float32), torch.tensor(self.dom[i])

def sample_weights(d):
    dw = d.domain.map(CFG["domain_weight"]).fillna(1.0).to_numpy(float)
    lab = d.label.to_numpy()
    tot = {c: dw[lab == c].sum() for c in np.unique(lab)}
    return dw / np.array([tot[c] for c in lab])

def make_loader(d, train):
    nw = CFG["num_workers"]
    kw = dict(batch_size=CFG["batch_size"], num_workers=nw, pin_memory=USE_AMP, persistent_workers=nw > 0)
    ds = PackDS(d, train_tf if train else eval_tf, train)
    if train:
        smp = WeightedRandomSampler(torch.as_tensor(sample_weights(d), dtype=torch.double), num_samples=len(d), replacement=True)
        return DataLoader(ds, sampler=smp, drop_last=len(d) > CFG["batch_size"], **kw)
    return DataLoader(ds, shuffle=False, **kw)

train_df = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "val"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)
assert len(val_df) and val_df.label.nunique() == 2, "Tập val cần có cả ảnh OK và ảnh Lỗi — hãy bổ sung dữ liệu."
train_loader, val_loader, test_loader = make_loader(train_df, True), make_loader(val_df, False), make_loader(test_df, False)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)} | {len(train_loader)} batch/epoch")

## 6. Mô hình

`PackagingNet` = backbone timm (bỏ lớp phân loại gốc) + 2 đầu tuyến tính:
- `head_bin` → 2 logit (OK, Lỗi), học bằng cross-entropy;
- `head_type` → logit đa nhãn cho từng loại lỗi, học bằng BCE **chỉ trên ảnh lỗi đã biết loại** (mặt nạ `type_known`).

Đầu vào là ảnh RGB giá trị 0–1; mean/std của backbone được **nhúng trong mô hình**. Ô benchmark dưới đo độ trễ ONNX trên CPU của Colab cho vài kiến trúc ứng viên để thấy vì sao chọn MobileNetV4 (CPU Colab nhanh hơn Pi 4 vài lần; số trên Pi đo lại bằng `benchmark_model.py`).

In [ ]:
import timm, onnxruntime as ort
from timm.data import resolve_model_data_config

def resolve_name(name, fallback):
    if not CFG["pretrained"]:
        return name.split(".")[0]
    avail = set(timm.list_models(pretrained=True))
    if name in avail:
        return name
    alts = sorted(m for m in avail if m.split(".")[0] == name.split(".")[0])
    pick = alts[0] if alts else fallback
    log.warning("Không có trọng số %s, dùng %s", name, pick)
    return pick

class PackagingNet(nn.Module):
    def __init__(self, name, n_types=NT, pretrained=True, drop=0.2):
        super().__init__()
        self.name = name
        self.backbone = timm.create_model(name, pretrained=pretrained, num_classes=0)
        dc = resolve_model_data_config(self.backbone)
        self.register_buffer("mean", torch.tensor(dc["mean"], dtype=torch.float32).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(dc["std"], dtype=torch.float32).view(1, 3, 1, 1), persistent=False)
        with torch.no_grad():
            self.backbone.eval()
            nf = self.pool(self.features(torch.zeros(1, 3, 128, 128))).shape[1]
        self.drop = nn.Dropout(drop)
        self.head_bin = nn.Linear(nf, 2)
        self.head_type = nn.Linear(nf, n_types)

    def features(self, x):                       # x: RGB 0..1, NCHW
        return self.backbone.forward_features((x - self.mean) / self.std)

    def pool(self, f):
        return self.backbone.forward_head(f, pre_logits=True)

    def forward(self, x):
        p = self.drop(self.pool(self.features(x)))
        return self.head_bin(p), self.head_type(p)

    def param_groups(self, lr, head_mult):
        head = list(self.head_bin.parameters()) + list(self.head_type.parameters())
        return [{"params": list(self.backbone.parameters()), "lr": lr}, {"params": head, "lr": lr * head_mult}]

class ExportWrapper(nn.Module):
    """Đầu ra cho Pi: probs = softmax(logit / T) [OK, Lỗi]; type_probs = sigmoid theo từng loại lỗi."""
    def __init__(self, m, temperature=1.0):
        super().__init__()
        self.m = m
        self.register_buffer("t", torch.tensor(float(temperature)))
    def forward(self, image):
        lb, lt = self.m(image)
        return torch.softmax(lb / self.t, 1), torch.sigmoid(lt)

def export_onnx(model, path, size, temperature=1.0):
    w = ExportWrapper(copy.deepcopy(model).cpu().eval(), temperature).eval()
    x = torch.rand(1, 3, size, size)
    kw = dict(input_names=["image"], output_names=["probs", "type_probs"], opset_version=CFG["onnx_opset"],
              dynamic_axes={"image": {0: "batch"}, "probs": {0: "batch"}, "type_probs": {0: "batch"}})
    errs = []
    for dyn in (False, True):                    # exporter cũ trước, exporter dynamo nếu bản PyTorch đã bỏ exporter cũ
        try:
            torch.onnx.export(w, (x,), str(path), dynamo=dyn, **kw)
            break
        except Exception as e:
            errs.append(repr(e)[:300])
    else:
        raise RuntimeError(f"Xuất ONNX thất bại: {errs}")
    import onnx
    onnx.checker.check_model(str(path))
    return path

def ort_session(path, threads=None):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads or min(4, os.cpu_count() or 1)
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])

def ort_latency(path, size, threads, runs=30):
    s = ort_session(path, threads)
    x = np.random.rand(1, 3, size, size).astype(np.float32)
    for _ in range(5):
        s.run(None, {"image": x})
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); s.run(None, {"image": x}); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts))

if CFG["run_zoo_bench"]:
    rows = []
    for nm in CFG["zoo_models"]:
        try:
            m = PackagingNet(nm, pretrained=False)
            p = export_onnx(m, CKPT / f"zoo_{nm}.onnx", S)
            rows.append(dict(model=nm, params_M=sum(t.numel() for t in m.parameters()) / 1e6,
                             ms_1_luong=ort_latency(p, S, 1), ms_da_luong=ort_latency(p, S, None)))
        except Exception as e:
            rows.append(dict(model=nm, error=str(e)[:150]))
    print(f"Độ trễ ONNX Runtime trên CPU máy hiện tại, ảnh {S}x{S}:")
    display(pd.DataFrame(rows).round(2))

## 7. Huấn luyện

Công thức chung cho cả thầy và trò:
- AdamW, learning rate đầu ra gấp `head_lr_mult` lần backbone, warmup 1 epoch rồi cosine;
- mixed precision (AMP) trên GPU, `channels_last`, clip gradient 1.0;
- **EMA trọng số** (bản trung bình trượt thường tổng quát tốt hơn), đánh giá EMA mỗi epoch;
- lưu bản tốt nhất theo **Average Precision lớp Lỗi** trên val, dừng sớm sau `patience` epoch không cải thiện.

Trò học thêm từ thầy: `loss = (1 − α)·CE(nhãn) + α·T²·KL(trò/T ‖ thầy/T) + λ·(BCE loại lỗi + 0.5·BCE với xác suất loại lỗi của thầy)`.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix

@torch.no_grad()
def predict(model, loader):
    model.eval()
    LB, LT, Y, TV, TM, D = [], [], [], [], [], []
    for x, y, tv, tm, dom in loader:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            lb, lt = model(x)
        LB.append(lb.float().cpu()); LT.append(lt.float().cpu())
        Y.append(y); TV.append(tv); TM.append(tm); D.append(dom)
    if not LB:
        return dict(lb=torch.zeros(0, 2), lt=torch.zeros(0, NT), y=torch.zeros(0, dtype=torch.long),
                    tv=torch.zeros(0, NT), tm=torch.zeros(0), dom=torch.zeros(0, dtype=torch.long))
    return dict(lb=torch.cat(LB), lt=torch.cat(LT), y=torch.cat(Y), tv=torch.cat(TV), tm=torch.cat(TM), dom=torch.cat(D))

def bin_metrics(y, p, thr=0.5):
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=float)
    pred = (p >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    rec, spec = tp / max(1, tp + fn), tn / max(1, tn + fp)
    prec = tp / max(1, tp + fp)
    out = dict(n=len(y), n_loi=int(y.sum()), accuracy=(tp + tn) / max(1, len(y)), balanced_acc=(rec + spec) / 2,
               recall_loi=rec, precision_loi=prec, specificity=spec, f1_loi=2 * prec * rec / max(1e-9, prec + rec),
               bo_sot_loi=fn, loai_nham_hang_tot=fp)
    both = len(set(y.tolist())) == 2
    out["auc"] = roc_auc_score(y, p) if both else float("nan")
    out["ap"] = average_precision_score(y, p) if both else float("nan")
    return out

def threshold_for_recall(y, p, target):
    """Ngưỡng LỚN NHẤT vẫn giữ recall lớp Lỗi >= target -> precision cao nhất có thể."""
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=float)
    pos = np.sort(p[y == 1])
    if len(pos) == 0:
        return 0.5
    k = int(np.floor((1 - target) * len(pos) + 1e-9))     # số ảnh lỗi được phép bỏ sót
    return float(np.nextafter(pos[min(k, len(pos) - 1)], -np.inf))

def type_loss(lt, tv, tm):
    l = F.binary_cross_entropy_with_logits(lt, tv, reduction="none").mean(1)
    return (l * tm).sum() / tm.sum().clamp(min=1.0)

def make_ema(model, total_steps):
    """Trả (ema, có_nhận_step). ModelEmaV3 cần số bước để khởi động decay từ thấp lên cao."""
    decay = min(CFG["ema_decay"], 1 - 5 / max(10, total_steps))
    try:
        from timm.utils import ModelEmaV3
        return ModelEmaV3(model, decay=decay, use_warmup=True), True
    except Exception:
        try:
            from timm.utils import ModelEmaV2
            return ModelEmaV2(model, decay=decay), False
        except Exception:
            return None, False

def fit(model, tag, epochs, lr, teacher=None):
    model = model.to(DEVICE).to(memory_format=torch.channels_last)
    if teacher is not None:
        teacher = teacher.to(DEVICE).to(memory_format=torch.channels_last).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
    opt = torch.optim.AdamW(model.param_groups(lr, CFG["head_lr_mult"]), weight_decay=CFG["weight_decay"])
    spe = len(train_loader); total = max(1, epochs * spe); warm = CFG["warmup_epochs"] * spe
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / max(1, warm) if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    ema, ema_step = make_ema(model, total)
    best, bad, hist, gstep = -1.0, 0, [], 0
    ckpt = CKPT / f"{tag}_best.pt"
    a, Tkd = CFG["kd_alpha"], CFG["kd_temperature"]
    for ep in range(epochs):
        model.train(); t0 = time.time(); run_loss, run_n = 0.0, 0
        for x, y, tv, tm, _ in tqdm(train_loader, desc=f"{tag} {ep + 1}/{epochs}", leave=False):
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            y, tv, tm = y.to(DEVICE), tv.to(DEVICE), tm.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                lb, lt = model(x)
                if teacher is not None:
                    with torch.no_grad():
                        tb, tt = teacher(x)
            lb, lt = lb.float(), lt.float()
            l_bin = F.cross_entropy(lb, y, label_smoothing=CFG["label_smoothing"])
            l_type = type_loss(lt, tv, tm)
            if teacher is None:
                loss = l_bin + CFG["type_loss_weight"] * l_type
            else:
                l_kd = F.kl_div(F.log_softmax(lb / Tkd, 1), F.softmax(tb.float() / Tkd, 1), reduction="batchmean") * Tkd * Tkd
                l_kdt = F.binary_cross_entropy_with_logits(lt, torch.sigmoid(tt.float()))
                loss = (1 - a) * l_bin + a * l_kd + CFG["type_loss_weight"] * (l_type + 0.5 * l_kdt)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            gstep += 1
            if ema is not None:
                ema.update(model, step=gstep) if ema_step else ema.update(model)
            run_loss += loss.item() * len(y); run_n += len(y)
        evm = ema.module if ema is not None else model
        out = predict(evm, val_loader)
        p = torch.softmax(out["lb"], 1)[:, 1].numpy(); yv = out["y"].numpy()
        m05 = bin_metrics(yv, p)
        mt = bin_metrics(yv, p, threshold_for_recall(yv, p, CFG["target_recall"]))
        score = m05["ap"] if not math.isnan(m05["ap"]) else m05["balanced_acc"]
        hist.append(dict(epoch=ep + 1, loss=run_loss / max(1, run_n), val_ap=m05["ap"], val_auc=m05["auc"],
                         val_bal_acc=m05["balanced_acc"], val_prec_at_target=mt["precision_loi"],
                         lr=opt.param_groups[0]["lr"], sec=time.time() - t0))
        print(f"[{tag}] epoch {ep + 1:02d} loss {hist[-1]['loss']:.4f} | val AP {m05['ap']:.4f} AUC {m05['auc']:.4f} "
              f"bal-acc {m05['balanced_acc']:.4f} | precision@recall{CFG['target_recall']:.0%} {mt['precision_loi']:.4f} "
              f"| {hist[-1]['sec']:.0f}s")
        if score > best + 1e-4:
            best, bad = score, 0
            torch.save({k: v.detach().cpu() for k, v in evm.state_dict().items()}, ckpt)
        else:
            bad += 1
            if bad >= CFG["patience"]:
                print(f"[{tag}] dừng sớm ở epoch {ep + 1}")
                break
    model.load_state_dict(torch.load(ckpt, map_location="cpu"))
    model.to(DEVICE).eval()
    h = pd.DataFrame(hist)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    h.plot(x="epoch", y="loss", ax=ax[0], title=f"{tag}: loss train", legend=False)
    h.plot(x="epoch", y=["val_ap", "val_auc", "val_bal_acc", "val_prec_at_target"], ax=ax[1], title=f"{tag}: chỉ số val")
    show_fig(fig, f"curves_{tag}")
    print(f"[{tag}] tốt nhất: val AP = {best:.4f}")
    return model, h

In [ ]:
# ---- 7.1 Train mô hình thầy ----
TEACHER_NAME = resolve_name(CFG["teacher_name"], "convnext_tiny.fb_in22k_ft_in1k")
teacher, hist_t = None, None
if CFG["use_teacher"]:
    teacher = PackagingNet(TEACHER_NAME, pretrained=CFG["pretrained"])
    print(f"Thầy: {TEACHER_NAME} — {sum(p.numel() for p in teacher.parameters()) / 1e6:.1f}M tham số")
    teacher, hist_t = fit(teacher, "teacher", CFG["teacher_epochs"], CFG["teacher_lr"])

In [ ]:
# ---- 7.2 Train mô hình trò (chưng cất từ thầy) ----
STUDENT_NAME = resolve_name(CFG["student_name"], "mobilenetv3_large_100.ra_in1k")
student = PackagingNet(STUDENT_NAME, pretrained=CFG["pretrained"])
print(f"Trò: {STUDENT_NAME} — {sum(p.numel() for p in student.parameters()) / 1e6:.1f}M tham số")
student, hist_s = fit(student, "student", CFG["student_epochs"], CFG["student_lr"], teacher=teacher)

rows = []
for nm, m in [("Thầy " + TEACHER_NAME, teacher), ("Trò " + STUDENT_NAME, student)]:
    if m is None:
        continue
    for split, ld in [("val", val_loader), ("test", test_loader)]:
        o = predict(m, ld)
        if len(o["y"]):
            rows.append({"mô hình": nm, "tập": split, **bin_metrics(o["y"].numpy(), torch.softmax(o["lb"], 1)[:, 1].numpy())})
display(pd.DataFrame(rows).round(4))
if teacher is not None:
    teacher.cpu()
    if USE_AMP:
        torch.cuda.empty_cache()

## 8. Hiệu chỉnh xác suất và chọn ngưỡng

1. **Temperature scaling** trên val: tìm T sao cho `softmax(logit / T)` khớp tần suất thật (xác suất 0.9 nghĩa là đúng khoảng 90 %). T được nhúng vào file ONNX.
2. **Ngưỡng quyết định:** lấy ngưỡng lớn nhất mà recall lớp Lỗi vẫn ≥ `target_recall` — ưu tiên không để lọt hàng hỏng, rồi mới tới việc ít loại nhầm hàng tốt. Nếu val có ≥ 20 ảnh lỗi tự chụp thì chọn ngưỡng trên phần ảnh tự chụp (đúng điều kiện trạm thật).

In [ ]:
def fit_temperature(logits, y):
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / log_t.exp(), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(log_t.exp().clamp(0.05, 20.0).item())

def ece(y, p, bins=10):
    """Expected Calibration Error của xác suất lớp Lỗi."""
    y, p = np.asarray(y), np.asarray(p)
    conf = np.where(p >= 0.5, p, 1 - p); pred = (p >= 0.5).astype(int)
    edges = np.linspace(0.5, 1.0, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

vo = predict(student, val_loader)
y_val, d_val = vo["y"].numpy(), vo["dom"].numpy()
T_CAL = fit_temperature(vo["lb"], vo["y"])
p_val_raw = torch.softmax(vo["lb"], 1)[:, 1].numpy()
p_val = torch.softmax(vo["lb"] / T_CAL, 1)[:, 1].numpy()
print(f"Nhiệt độ T = {T_CAL:.3f} | ECE trước {ece(y_val, p_val_raw):.4f} → sau {ece(y_val, p_val):.4f}")

own_v = d_val == DOM2ID["own"]
THR_SCOPE = "ảnh tự chụp" if (y_val[own_v] == 1).sum() >= 20 else "toàn bộ val"
sel = own_v if THR_SCOPE == "ảnh tự chụp" else np.ones_like(own_v, bool)
THR = threshold_for_recall(y_val[sel], p_val[sel], CFG["target_recall"])
if (y_val[sel] == 1).sum() < 30:
    log.warning("Val chỉ có %d ảnh lỗi — ngưỡng còn kém tin cậy, nên bổ sung ảnh lỗi tự chụp.", int((y_val[sel] == 1).sum()))
print(f"Ngưỡng DEFECT = {THR:.4f} (chọn trên {THR_SCOPE}, mục tiêu recall lỗi ≥ {CFG['target_recall']:.0%})")
display(pd.DataFrame([bin_metrics(y_val[sel], p_val[sel], THR)]).round(4))

## 9. Đánh giá trên tập test
Số liệu cho slide nằm ở dòng **"Ảnh tự chụp"**. Dòng "Công khai" cho biết mô hình có tổng quát ra ngoài trạm không.

In [ ]:
to = predict(student, test_loader)
y_te, d_te = to["y"].numpy(), to["dom"].numpy()
p_te = torch.softmax(to["lb"] / T_CAL, 1)[:, 1].numpy()
tp_te = torch.sigmoid(to["lt"]).numpy()

TEST_METRICS = {}
for nm, mask in [("Tất cả", np.ones_like(y_te, bool)), ("Ảnh tự chụp", d_te == DOM2ID["own"]),
                 ("Công khai", d_te == DOM2ID["public"])]:
    if mask.sum():
        TEST_METRICS[nm] = bin_metrics(y_te[mask], p_te[mask], THR)
display(pd.DataFrame(TEST_METRICS).T.round(4))

def plot_cm(ax, y, pred, title):
    cm = confusion_matrix(y, pred, labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=13, color="black" if v < cm.max() / 2 else "white")
    ax.set_xticks([0, 1], ["OK", "Lỗi"]); ax.set_yticks([0, 1], ["OK", "Lỗi"])
    ax.set_xlabel("Dự đoán"); ax.set_ylabel("Thật"); ax.set_title(title, fontsize=10)

if len(y_te):
    fig, ax = plt.subplots(1, 4, figsize=(19, 4))
    plot_cm(ax[0], y_te, (p_te >= THR).astype(int), f"Ma trận nhầm lẫn — tất cả (ngưỡng {THR:.3f})")
    ow = d_te == DOM2ID["own"]
    if ow.any():
        plot_cm(ax[1], y_te[ow], (p_te[ow] >= THR).astype(int), "Ma trận nhầm lẫn — ảnh tự chụp")
    else:
        ax[1].axis("off")
    if len(set(y_te.tolist())) == 2:
        fpr, tpr, _ = roc_curve(y_te, p_te); ax[2].plot(fpr, tpr); ax[2].plot([0, 1], [0, 1], "--", c="gray")
        ax[2].set_title(f"ROC (AUC {roc_auc_score(y_te, p_te):.3f})"); ax[2].set_xlabel("Tỉ lệ loại nhầm hàng tốt"); ax[2].set_ylabel("Recall lỗi")
        pr, rc, _ = precision_recall_curve(y_te, p_te); ax[3].plot(rc, pr)
        m = TEST_METRICS["Tất cả"]; ax[3].scatter([m["recall_loi"]], [m["precision_loi"]], c="red", zorder=3, label="điểm làm việc")
        ax[3].set_title(f"Precision–Recall (AP {average_precision_score(y_te, p_te):.3f})"); ax[3].set_xlabel("Recall lỗi"); ax[3].set_ylabel("Precision lỗi"); ax[3].legend()
    show_fig(fig, "test_evaluation")

# ---- Đầu ra loại lỗi: AP từng loại trên ảnh lỗi đã biết loại ----
known = (to["tm"].numpy() > 0) & (y_te == 1)
TYPE_AP = {}
for i, t in enumerate(TYPES):
    yt = to["tv"].numpy()[known, i]
    if known.sum() and 0 < yt.sum() < len(yt):
        TYPE_AP[t] = float(average_precision_score(yt, tp_te[known, i]))
print("AP theo loại lỗi (test):", {f"{t} ({TYPE_VI.get(t, t)})": round(v, 3) for t, v in TYPE_AP.items()} or "chưa đủ ảnh có nhãn loại")

In [ ]:
# ---- Ảnh dự đoán sai + Grad-CAM (mô hình nhìn vào đâu) ----
def gradcam(model, x, cls=1):
    model.eval()
    f = model.features(x)
    f.retain_grad()
    logit = model.head_bin(model.pool(f))[:, cls].sum()
    model.zero_grad(set_to_none=True)
    logit.backward()
    w = f.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * f).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    cam -= cam.amin(dim=(1, 2), keepdim=True)
    return (cam / cam.amax(dim=(1, 2), keepdim=True).clamp(min=1e-6)).detach().cpu().numpy()

def load_eval(path):
    return eval_tf(image=np.asarray(Image.open(path).convert("RGB")))["image"]

if len(y_te):
    wrong = np.where((p_te >= THR).astype(int) != y_te)[0][:24]
    show_grid([test_df.path[i] for i in wrong],
              [f"thật {'Lỗi' if y_te[i] else 'OK'} | p_lỗi {p_te[i]:.2f}" for i in wrong], ncol=8, name="misclassified")

    pick = list(np.where(y_te == 1)[0][:8]) + list(np.where(y_te == 0)[0][:4])
    imgs = [load_eval(test_df.path[i]) for i in pick]
    xb = torch.stack([torch.from_numpy(im.transpose(2, 0, 1).copy()).float() / 255 for im in imgs])
    sm = copy.deepcopy(student).float().to(DEVICE).eval()
    cams = gradcam(sm, xb.to(DEVICE))
    over = [(0.55 * im + 0.45 * plt.cm.jet(c)[..., :3] * 255).astype(np.uint8) for im, c in zip(imgs, cams)]
    show_grid(over, [f"{'Lỗi' if y_te[i] else 'OK'} | p {p_te[i]:.2f}" for i in pick], ncol=6, name="gradcam", size=2.6)
    del sm

## 10. Xuất ONNX cho Raspberry Pi

1. Xuất **FP32** (đã nhúng chuẩn hóa và nhiệt độ T), so sánh đầu ra với PyTorch (sai số phải < 1e-4).
2. **Lượng tử hóa tĩnh INT8** (QDQ, theo từng kênh, hiệu chuẩn bằng ảnh val).
3. Đánh giá cả hai bằng ONNX Runtime trên test, đo độ trễ; **chỉ chọn INT8 nếu AP giảm ≤ 0.01 và recall lỗi giảm ≤ 0.02**.

In [ ]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

stu_cpu = copy.deepcopy(student).float().cpu().eval()
FP32 = export_onnx(stu_cpu, OUT / "packaging_cls_fp32.onnx", S, T_CAL)

xb = next(iter(test_loader if len(test_df) else val_loader))[0][:8]
with torch.no_grad():
    ref = ExportWrapper(stu_cpu, T_CAL).eval()(xb)
got = ort_session(FP32).run(None, {"image": xb.numpy()})
diff = max(float(np.abs(ref[0].numpy() - got[0]).max()), float(np.abs(ref[1].numpy() - got[1]).max()))
print(f"Sai lệch PyTorch ↔ ONNX FP32: {diff:.2e}")
assert diff < 1e-4, "ONNX khác PyTorch quá nhiều"

def ort_predict(path, loader):
    s = ort_session(path)
    P, Y = [], []
    for x, y, *_ in loader:
        P.append(s.run(None, {"image": x.numpy()})[0][:, 1]); Y.append(y.numpy())
    return (np.concatenate(P), np.concatenate(Y)) if P else (np.zeros(0), np.zeros(0, int))

class CalibReader(CalibrationDataReader):
    def __init__(self, loader, n=200):
        xs = []
        for x, *_ in loader:
            xs += [x[i:i + 1].numpy() for i in range(len(x))]
            if len(xs) >= n:
                break
        self._it = iter([{"image": a} for a in xs[:n]])
    def get_next(self):
        return next(self._it, None)

INT8 = OUT / "packaging_cls_int8.onnx"
int8_ok = False
try:
    src = FP32
    try:
        from onnxruntime.quantization.shape_inference import quant_pre_process
        pre = CKPT / "fp32_pre.onnx"
        quant_pre_process(str(FP32), str(pre))
        src = pre
    except Exception as e:
        log.warning("Bỏ qua bước tiền xử lý lượng tử: %s", e)
    quantize_static(str(src), str(INT8), CalibReader(val_loader), quant_format=QuantFormat.QDQ, per_channel=True,
                    activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
    int8_ok = True
except Exception as e:
    log.warning("Lượng tử INT8 thất bại, dùng FP32: %s", e)

cmp_rows, CAND = [], {"fp32": FP32}
if int8_ok:
    CAND["int8"] = INT8
for kind, path in CAND.items():
    pv, yv = ort_predict(path, val_loader)
    thr_k = threshold_for_recall(yv[sel], pv[sel], CFG["target_recall"])
    pt, yt = ort_predict(path, test_loader)
    m = bin_metrics(yt, pt, thr_k) if len(yt) else bin_metrics(yv, pv, thr_k)
    cmp_rows.append(dict(ban=kind, nguong=thr_k, MB=path.stat().st_size / 2**20,
                         ms_1_luong=ort_latency(path, S, 1), ms_da_luong=ort_latency(path, S, None),
                         ap=m["ap"], recall_loi=m["recall_loi"], precision_loi=m["precision_loi"], balanced_acc=m["balanced_acc"]))
cmp = pd.DataFrame(cmp_rows).set_index("ban")
display(cmp.round(4))

DEPLOY = "fp32"
if "int8" in cmp.index:
    f, q = cmp.loc["fp32"], cmp.loc["int8"]
    ap_ok = math.isnan(f.ap) or q.ap >= f.ap - 0.01
    if ap_ok and q.recall_loi >= f.recall_loi - 0.02:
        DEPLOY = "int8"
THR_DEPLOY = float(cmp.loc[DEPLOY, "nguong"])
shutil.copy(CAND[DEPLOY], OUT / "packaging_cls.onnx")
print(f"Chọn bản {DEPLOY.upper()} → {OUT / 'packaging_cls.onnx'} | ngưỡng {THR_DEPLOY:.4f}")

In [ ]:
# ---- Ghi model_card.json, labels.json, metrics.json và ví dụ suy luận cho Pi ----
def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if math.isnan(float(o)) else round(float(o), 6)
    if isinstance(o, (np.integer,)):
        return int(o)
    return o

counts = {f"{d}_{'loi' if l else 'ok'}": int(n) for (d, l), n in df.groupby(["domain", "label"]).size().items()}
card = {
    "model_name": "packaging_cls", "version": CFG["model_version"],
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "architecture": STUDENT_NAME, "teacher": TEACHER_NAME if teacher is not None else None,
    "distilled": teacher is not None, "precision": DEPLOY, "file": "packaging_cls.onnx",
    "input": {"name": "image", "shape": [1, 3, S, S], "size": S, "dtype": "float32", "layout": "NCHW", "color": "RGB",
              "range": [0.0, 1.0], "resize": f"cắt ROI rồi resize thẳng về {S}x{S} bằng cv2.INTER_AREA",
              "normalization": "đã nhúng trong mô hình — không trừ mean/chia std thêm"},
    "outputs": {"probs": ["OK", "DEFECT"], "type_probs": TYPES},
    "decision_rule": "DEFECT nếu probs[0][1] >= defect_threshold, ngược lại OK",
    "defect_threshold": THR_DEPLOY, "temperature": T_CAL, "target_recall": CFG["target_recall"], "threshold_chosen_on": THR_SCOPE,
    "damage_types": TYPES, "damage_types_vi": {t: TYPE_VI.get(t, t) for t in TYPES}, "type_threshold": 0.5,
    "metrics": {"test": TEST_METRICS, "type_ap_test": TYPE_AP, "onnx_compare": cmp.reset_index().to_dict("records")},
    "data": {"counts": counts, "sources": df.source.value_counts().to_dict(),
             "split": {k: int(v) for k, v in df.split.value_counts().items()}},
    "train_config": {k: CFG[k] for k in ["img_size", "teacher_epochs", "student_epochs", "teacher_lr", "student_lr",
                                         "kd_alpha", "kd_temperature", "synth_defect_prob", "domain_weight"]},
}
(OUT / "model_card.json").write_text(json.dumps(jsonable(card), ensure_ascii=False, indent=2))
(OUT / "labels.json").write_text(json.dumps({"classes": ["OK", "DEFECT"], "types": TYPES}, indent=2))
(OUT / "metrics.json").write_text(json.dumps(jsonable({"test": TEST_METRICS, "type_ap": TYPE_AP}), ensure_ascii=False, indent=2))
torch.save({"arch": STUDENT_NAME, "state_dict": stu_cpu.state_dict(), "types": TYPES, "img_size": S}, OUT / "student_best.pt")

PI_EXAMPLE = """# Ví dụ suy luận trên Raspberry Pi — chép cùng thư mục với packaging_cls.onnx và model_card.json
import json, time, cv2, numpy as np, onnxruntime as ort

card = json.load(open("model_card.json", encoding="utf-8"))
S, THR = card["input"]["size"], card["defect_threshold"]
so = ort.SessionOptions(); so.intra_op_num_threads = 4
sess = ort.InferenceSession(card["file"], so, providers=["CPUExecutionProvider"])

def predict(bgr_roi):
    x = cv2.resize(cv2.cvtColor(bgr_roi, cv2.COLOR_BGR2RGB), (S, S), interpolation=cv2.INTER_AREA)
    x = (x.astype(np.float32) / 255.0).transpose(2, 0, 1)[None]
    t0 = time.perf_counter()
    probs, type_probs = sess.run(None, {"image": x})
    p_defect = float(probs[0, 1])
    types = {t: float(v) for t, v in zip(card["damage_types"], type_probs[0])}
    label = "DEFECT" if p_defect >= THR else "OK"
    return label, p_defect, types, (time.perf_counter() - t0) * 1000

if __name__ == "__main__":
    import sys
    print(predict(cv2.imread(sys.argv[1])))
"""
(OUT / "pi_predict_example.py").write_text(PI_EXAMPLE)

for f in FIG.glob("*.png"):
    shutil.copy(f, OUT / f.name)
shutil.copy(WORK / "manifest_split.csv", OUT / "manifest_split.csv")
ZIP = shutil.make_archive(str(WORK / "packaging_ai_export"), "zip", OUT)
if CFG["use_drive"] and IN_COLAB:
    dst = Path(CFG["drive_dir"]) / f"export_{datetime.datetime.now():%Y%m%d_%H%M}"
    shutil.copytree(OUT, dst)
    print("Đã lưu lên Drive:", dst)
print("Gói kết quả:", ZIP)
print(sorted(p.name for p in OUT.iterdir()))
if SMOKE:
    for need in ["packaging_cls.onnx", "model_card.json", "labels.json", "pi_predict_example.py"]:
        assert (OUT / need).exists(), need
    print("SMOKE TEST: toàn bộ pipeline chạy xong.")

## 11. Dùng mô hình trên Raspberry Pi

1. Chép 3 file `packaging_cls.onnx`, `model_card.json`, `labels.json` vào `/opt/sorter/models/` trên Pi.
2. Trong `/etc/sorter/config.yaml` đặt `ai.backend: onnx`, `ai.model_path`, `ai.labels_path`; **`ai.input_size` lấy theo `model_card.json`** (mặc định notebook dùng 256) — hoặc để `OnnxClassifier` đọc thẳng `model_card.json`.
3. Quy tắc quyết định: `DEFECT` nếu `probs[0][1] >= defect_threshold` (ngưỡng đã được chọn để recall lỗi ≥ 95 %). Không chuẩn hóa mean/std trên Pi vì đã nhúng trong mô hình.
4. Đo tốc độ thật bằng `pi/tools/benchmark_model.py`, chạy thử `python pi_predict_example.py anh_hop.jpg`.

### Vòng cải thiện (active learning)
- Trên Pi, ảnh bị loại do lỗi được lưu ở `/var/lib/sorter/defects/`. Định kỳ chép về, xem lại ảnh dự đoán sai (đặc biệt **lỗi bị bỏ sót**), gán nhãn đúng vào thư mục ảnh tự chụp rồi chạy lại notebook.
- Mỗi lần thêm ảnh mới, tăng `model_version` trong cấu hình để dashboard hiện đúng phiên bản.
- Nếu `test_own` còn thấp mà `test_public` cao: thiếu ảnh tự chụp, hoặc đèn/góc chụp khi thu dữ liệu khác lúc chạy thật.